In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  CSPF PHASE 1 — FINAL PRODUCTION CODE                                      ║
║  Complete Perturbation Generation & Semantic Metrics                         ║
║                                                                              ║
║  • 12 Perturbation Families (P01-P12)                                       ║
║  • 3 Severity Levels (mild, medium, strong) per family                      ║
║  • Grammar-aware smart position selection                                    ║
║  • 8 Semantic preservation metrics (M1-M8)                                  ║
║  • Full reproducibility via deterministic seeding                            ║
║  • Auto-save to Google Drive                                                ║
║                                                                              ║
║  Run in Google Colab:                                                        ║
║  1. Mount Drive: !gdown setup in Colab                                      ║
║  2. Upload datasets to datasets/ folder                                     ║
║  3. Run this script                                                          ║
║  4. Results auto-save to:                                                    ║
║     /My Drive/postdoc-papers/paper2/generated files phase1/                ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os, re, sys, glob, random, warnings, logging, hashlib, gc
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

print("\n" + "╔" + "═"*78 + "╗")
print("║" + " CSPF PHASE 1 — COMPLETE IMPLEMENTATION ".center(78) + "║")
print("╚" + "═"*78 + "╝\n")

# ═════════════════════════════════════════════════════════════════════════════
# SETUP & GOOGLE DRIVE
# ═════════════════════════════════════════════════════════════════════════════

# Try to mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_PATH = "/content/drive/MyDrive/postdoc-papers/paper2/generated files phase1"
    os.makedirs(GDRIVE_PATH, exist_ok=True)
    SAVE_TO_GDRIVE = True
    print(f"✓ Google Drive mounted")
    print(f"✓ Output path: {GDRIVE_PATH}\n")
except:
    GDRIVE_PATH = None
    SAVE_TO_GDRIVE = False
    print(f"⚠ Not in Colab — will save to local output/ directory\n")

# Local backup
LOCAL_OUTPUT = "output"
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

INPUT_DIR = "/content/drive/MyDrive/postdoc-papers/paper2/datasets/"
os.makedirs(INPUT_DIR, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTS & REPRODUCIBILITY
# ═════════════════════════════════════════════════════════════════════════════

GLOBAL_SEED = 42
SEVERITY = ("mild", "medium", "strong")

def _seed(review_idx: int, family: str, severity: str, position: str) -> int:
    """Deterministic seed for reproducibility."""
    key = f"{GLOBAL_SEED}|{review_idx}|{family}|{severity}|{position}"
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)

def _pick(pool: list, review_idx: int, family: str, severity: str, position: str) -> str:
    """Reproducible selection from severity-stratified pool."""
    rng = random.Random(_seed(review_idx, family, severity, position))
    return rng.choice(list(pool))

# ═════════════════════════════════════════════════════════════════════════════
# SENTIMENT ADJECTIVES & INTENSIFIERS
# ═════════════════════════════════════════════════════════════════════════════

POS_ADJS = ["good","great","excellent","amazing","wonderful","fantastic","brilliant",
    "superb","outstanding","perfect","enjoyable","pleasant","delightful","impressive",
    "splendid","marvelous","terrific","awesome","exceptional","fabulous","magnificent",
    "stunning","remarkable","incredible","beautiful","lovely","charming","captivating",
    "engaging","entertaining","satisfying","rewarding","fulfilling","heartwarming",
    "inspiring","moving","touching","compelling","riveting","gripping","thrilling",
    "exciting","refreshing","innovative","creative","clever","witty","smart",
    "thoughtful","meaningful","powerful"]

NEG_ADJS = ["bad","terrible","awful","horrible","dreadful","poor","mediocre",
    "disappointing","boring","dull","tedious","unpleasant","irritating","frustrating",
    "annoying","offensive","disturbing","pathetic","ridiculous","absurd","atrocious",
    "appalling","disgusting","repulsive","nauseating","abysmal","catastrophic",
    "disastrous","deplorable","shameful","miserable","wretched","lousy","rotten",
    "pointless","worthless","useless","unbearable","insufferable","intolerable",
    "infuriating","maddening","sickening"]

ALL_ADJS = POS_ADJS + NEG_ADJS
_ADJ_PAT = '|'.join(re.escape(a) for a in sorted(ALL_ADJS, key=len, reverse=True))

INTENSIFIERS = ["absolutely","completely","entirely","extremely","highly","incredibly",
    "insanely","intensely","outrageously","overwhelmingly","particularly","profoundly",
    "purely","remarkably","seriously","severely","significantly","so","strikingly",
    "super","terribly","thoroughly","totally","tremendously","truly","unbelievably",
    "utterly","very","wildly","exceptionally","extraordinarily","fantastically",
    "genuinely","immensely","magnificently","passionately","phenomenally",
    "spectacularly","stupendously","sublimely","supremely","unquestionably",
    "undeniably","unreservedly","wholeheartedly"]

_INT_PAT = '|'.join(re.escape(i) for i in sorted(INTENSIFIERS, key=len, reverse=True))

# ═════════════════════════════════════════════════════════════════════════════
# SMART POSITION DETECTORS (grammar-aware)
# ═════════════════════════════════════════════════════════════════════════════

def _find_predicate_adj(text: str):
    """Find main sentiment adjective: copula+adj, or bare adj."""
    for pat in [
        r'\b(?:is|was|are|were|feels?|seems?|looks?|sounds?|appears?)\s+'
        r'(?:(?:' + _INT_PAT + r')\s+){0,2}(' + _ADJ_PAT + r')\b',
        r'\b(?:find|found|consider(?:ed)?|thought|felt)\s+it\s+'
        r'(?:(?:' + _INT_PAT + r')\s+){0,2}(' + _ADJ_PAT + r')\b',
        r'\b(' + _ADJ_PAT + r')\b',
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            g = m.lastindex or 1
            try: return m.start(g), m.end(g), m.group(g)
            except: pass
    return None, None, None

def _find_intensifier_adj_bigram(text: str):
    """Find INTENSIFIER + ADJECTIVE bigram."""
    for inten in sorted(INTENSIFIERS, key=len, reverse=True):
        pat = r'\b(' + re.escape(inten) + r')\s+(' + _ADJ_PAT + r')\b'
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.start(1), m.end(1), m.start(2), m.end(2), m.group(1), m.group(2)
    return None, None, None, None, None, None

def _find_clause_boundary(text: str, start: int) -> int:
    """Find clause boundary (conjunction or punctuation) after position."""
    tail = text[start:]
    m = re.search(r'(?=\s+(?:and|but|or|although|because|however|yet|while)\b|\s*[.!?]\s*|$)', tail)
    return start + (m.start() if m else len(tail))

def _strip_end(text: str) -> str:
    return text.rstrip(".,!? ")

def _lower_first(text: str) -> str:
    return text[0].lower() + text[1:] if text and text[0].isupper() else text

# ═════════════════════════════════════════════════════════════════════════════
# EXPRESSION POOLS (severity-stratified per family)
# ═════════════════════════════════════════════════════════════════════════════

HEDGE_POS = {
    "mild":   ["quite","fairly","rather","reasonably","pretty","somewhat"],
    "medium": ["somewhat","relatively","moderately","arguably","seemingly","in a way"],
    "strong": ["ostensibly","purportedly","in a sense","not entirely","by some measures","nominally"],
}
HEDGE_NEG = {
    "mild":   ["fairly","quite","rather","relatively","somewhat","to some extent"],
    "medium": ["not really","not particularly","not especially","not terribly","not exactly","sort of not"],
    "strong": ["ostensibly","purportedly","arguably","not entirely","in some respects","nominally not"],
}
HEDGE_CLAUSE_FIN = {
    "mild":   ["I think","more or less","in a way","to some degree","I'd say","so to speak"],
    "medium": ["I suppose","I guess","for the most part","to varying degrees","within reason","arguably"],
    "strong": ["arguably","one might say","for what it's worth","to put it mildly","if that","in theory"],
}
HEDGE_SENT_INIT = {
    "mild":   ["To be fair,","Honestly,","In my view,","I'd say","Speaking personally,","In my opinion,"],
    "medium": ["Arguably,","Seemingly,","For what it's worth,","In all fairness,","One could say","Apparently,"],
    "strong": ["Ostensibly,","Purportedly,","By most accounts,","Nominally speaking,","In theory,","Technically,"],
}

DISCOURSE = {
    "mild":   ["Honestly,","Frankly,","To be fair,","In my experience,","Speaking for myself,","Truth be told,"],
    "medium": ["Surprisingly,","Interestingly enough,","For what it's worth,","All things considered,","On reflection,","Notably,"],
    "strong": ["Objectively speaking,","Technically speaking,","Upon closer inspection,","Against all expectations,","Contrary to expectations,","Setting aside bias,"],
}

NEG_SOFT_POS = {
    "mild":   {"good":"not bad","great":"not poor","excellent":"not substandard","amazing":"not unimpressive","wonderful":"not unpleasant"},
    "medium": {"good":"not terrible","great":"not dreadful","excellent":"not inferior","amazing":"not unremarkable","wonderful":"not distasteful"},
    "strong": {"good":"not exactly great","great":"not quite perfect","excellent":"not without its flaws","amazing":"not beyond question","wonderful":"not universally praised"},
}
NEG_SOFT_NEG = {
    "mild":   {"bad":"not good","terrible":"not enjoyable","awful":"not pleasant","horrible":"not appealing","dreadful":"not satisfying"},
    "medium": {"bad":"not that good","terrible":"not at all enjoyable","awful":"not at all pleasant","horrible":"not the least appealing","dreadful":"hardly satisfying"},
    "strong": {"bad":"not exactly a success","terrible":"not what anyone would call enjoyable","awful":"not something to recommend","horrible":"not in the least redeeming"},
}

SARCASM_PARENTH = {
    "mild":   ["(I suppose)","(more or less)","(in theory)","(at least, that's the idea)","(supposedly)"],
    "medium": ["(apparently)","(or so it seemed)","(believe it or not)","(go figure)","(one could say)"],
    "strong": ["(what a revelation)","(shocking, I know)","(color me surprised)","(how unexpected)","(surprise surprise)"],
}
SARCASM_CLAUSE_FIN = {
    "mild":   [", I suppose",", more or less",", in a way",", so they say",", they say"],
    "medium": [", apparently",", or so it would seem",", believe it or not",", go figure",", supposedly"],
    "strong": [", what a surprise",", I'm shocked",", who would have thought",", truly groundbreaking",", shocking"],
}

DILUTION = {
    "mild":   ["quite","fairly","pretty","rather","reasonably","somewhat","sort of","kind of"],
    "medium": ["somewhat","moderately","mildly","slightly","a bit","fairly","rather"],
    "strong": ["barely","hardly","only marginally","just barely","at best slightly","scarcely"],
}

PRAGMATIC_POS = {
    "mild":   {"it is amazing":"it is quite good","it was amazing":"it was quite good","it is excellent":"it is solid"},
    "medium": {"it is amazing":"it performs well","it was amazing":"it performed well","it is excellent":"it delivers"},
    "strong": {"it is amazing":"it is not without merit","it was amazing":"it was not without merit","it is excellent":"it has its moments"},
}
PRAGMATIC_NEG = {
    "mild":   {"it is terrible":"it is problematic","it was terrible":"it was problematic","it is awful":"it is flawed"},
    "medium": {"it is terrible":"it fails to deliver","it was terrible":"it failed to deliver","it is awful":"it does not work"},
    "strong": {"it is terrible":"it is fundamentally broken","it was terrible":"it was fundamentally broken","it is awful":"it offers nothing of value"},
}

CONTRAST_POS = {
    "mild":   {"good":"not terrible","great":"not dreadful","excellent":"not inferior","amazing":"not unimpressive"},
    "medium": {"good":"not exactly bad","great":"not quite poor","excellent":"not really subpar","amazing":"not entirely forgettable"},
    "strong": {"good":"not the worst","great":"not altogether awful","excellent":"not entirely without problems"},
}
CONTRAST_NEG = {
    "mild":   {"terrible":"not amazing","awful":"not pleasant","horrible":"not appealing","bad":"not good"},
    "medium": {"terrible":"not exactly brilliant","awful":"not quite good","horrible":"not really appealing"},
    "strong": {"terrible":"not what anyone would call good","awful":"not remotely pleasant","horrible":"not even close to acceptable"},
}

SYNONYM_POS = {
    "mild":   {"amazing":"wonderful","wonderful":"amazing","great":"excellent","excellent":"great","fantastic":"brilliant","brilliant":"fantastic"},
    "medium": {"amazing":"spectacular","wonderful":"magnificent","great":"splendid","excellent":"exceptional","fantastic":"phenomenal"},
    "strong": {"amazing":"first-rate","wonderful":"awe-inspiring","great":"top-notch","excellent":"unparalleled"},
}
SYNONYM_NEG = {
    "mild":   {"bad":"poor","terrible":"dreadful","awful":"dreadful","horrible":"dreadful","poor":"inferior"},
    "medium": {"bad":"substandard","terrible":"appalling","awful":"appalling","horrible":"appalling","poor":"inadequate"},
    "strong": {"bad":"deplorable","terrible":"atrocious","awful":"abysmal","horrible":"ghastly","poor":"deficient"},
}

INTENSITY_BY_STRENGTH = {
    "mild":   ["quite","fairly","pretty","rather","reasonably","somewhat","particularly","notably"],
    "medium": ["very","really","truly","genuinely","seriously","significantly","considerably","substantially"],
    "strong": ["absolutely","completely","extremely","incredibly","utterly","totally","entirely","thoroughly"],
}

SYNTAX_PATTERNS = {
    "mild": [
        (r'\bI liked\b', "It was likable"),
        (r'\bI loved\b', "It was lovable"),
        (r'\bI enjoyed\b', "It was enjoyable"),
        (r'\bI hated\b', "It was detestable"),
    ],
    "medium": [
        (r'\bI thought it was\b', "It was considered"),
        (r'\bI consider it\b', "It is considered"),
    ],
    "strong": [
        (r'\bThis movie is\b', "The movie proves to be"),
        (r'\bThis film is\b', "The film proves to be"),
    ],
}

ENTITY_MAP = {
    "movie": (["film","picture"],["production","feature"],["cinematic work","motion picture"]),
    "film": (["movie","picture"],["production","feature"],["cinematic work","motion picture"]),
    "book": (["novel","volume"],["publication","title"],["literary work","written work"]),
    "show": (["series","program"],["production","presentation"],["television series","broadcast"]),
    "game": (["title","release"],["product","experience"],["interactive work","software title"]),
}

REGISTER_BY_SEV = {
    "mild": {"pretty good":"decent","kind of good":"reasonably good","not bad":"acceptable","okay":"fine"},
    "medium": {"fire":"exceptional","lit":"outstanding","dope":"impressive","gem":"exceptional find","goat":"outstanding"},
    "strong": {"this is fire":"this is exceptional","slaps":"resonates strongly","trash":"of poor quality"},
}

# ═════════════════════════════════════════════════════════════════════════════
# PERTURBATION FUNCTIONS (ALL 12 FAMILIES)
# ═════════════════════════════════════════════════════════════════════════════

def _rec(p, sev, pos, ling, label):
    return {"perturbed_review":p, "severity":sev, "position_strategy":pos,
            "linguistic_target":ling, "variant_label":label}

def p01_hedging(text, sentiment, ridx):
    """P01: Hedging — prepend cautious markers at 3 valid positions."""
    adj_s, adj_e, adj_w = _find_predicate_adj(text)
    pools = HEDGE_POS if sentiment=="positive" else HEDGE_NEG
    results = []

    for sev in SEVERITY:
        # Position 1: pre-adjective
        if adj_s is not None:
            expr = _pick(pools[sev], ridx, "P01", sev, "pre-adjective")
            pert = text[:adj_s] + expr + " " + text[adj_s:]
            results.append(_rec(pert, sev, "pre-adjective", "adjective", f"hedge={expr}"))

        # Position 2: clause-final
        expr = _pick(HEDGE_CLAUSE_FIN[sev], ridx, "P01", sev, "clause-final")
        pert = _strip_end(text) + ", " + expr + "."
        results.append(_rec(pert, sev, "clause-final", "clause", f"hedge={expr}"))

        # Position 3: sentence-initial
        expr = _pick(HEDGE_SENT_INIT[sev], ridx, "P01", sev, "sentence-initial")
        sep = " " if expr.endswith(",") else ", "
        pert = expr + sep + _lower_first(text)
        results.append(_rec(pert, sev, "sentence-initial", "clause", f"hedge={expr}"))

    return results

def p02_discourse(text, sentiment, ridx):
    """P02: Discourse Injection — sentence-initial only."""
    results = []
    for sev in SEVERITY:
        marker = _pick(DISCOURSE[sev], ridx, "P02", sev, "sentence-initial")
        pert = marker + " " + _lower_first(text)
        results.append(_rec(pert, sev, "sentence-initial", "clause", f"discourse={marker}"))
    return results

def p03_negation(text, sentiment, ridx):
    """P03: Negation Softening — adjective-replacement only."""
    pools = NEG_SOFT_POS if sentiment=="positive" else NEG_SOFT_NEG
    adj_s, adj_e, adj_w = _find_predicate_adj(text)
    results = []

    for sev in SEVERITY:
        mapping = pools[sev]
        target = adj_w.lower() if adj_w else None
        repl = mapping.get(target) if target else None
        if repl is None: continue
        pert = text[:adj_s] + repl + text[adj_e:]
        results.append(_rec(pert, sev, "adjective-replacement", "adjective", f"{target}→{repl}"))

    return results

def p04_sarcasm(text, sentiment, ridx):
    """P04: Sarcasm-Lite — parenthetical and clause-final."""
    adj_s, adj_e, adj_w = _find_predicate_adj(text)
    results = []

    for sev in SEVERITY:
        # Position 1: parenthetical
        if adj_e is not None:
            marker = _pick(SARCASM_PARENTH[sev], ridx, "P04", sev, "parenthetical")
            pert = text[:adj_e] + " " + marker + text[adj_e:]
            results.append(_rec(pert, sev, "parenthetical", "adjective", f"sarcasm={marker}"))

        # Position 2: clause-final
        marker = _pick(SARCASM_CLAUSE_FIN[sev], ridx, "P04", sev, "clause-final")
        pert = _strip_end(text) + marker + "."
        results.append(_rec(pert, sev, "clause-final", "clause", f"sarcasm={marker}"))

    return results

def p05_dilution(text, sentiment, ridx):
    """P05: Emotional Dilution — adverb-weakening of intensifier+adj."""
    i_s, i_e, a_s, a_e, inten, adj_w = _find_intensifier_adj_bigram(text)
    results = []

    for sev in SEVERITY:
        weaker = _pick(DILUTION[sev], ridx, "P05", sev, "adverb-weakening")
        if i_s is not None:
            pert = text[:i_s] + weaker + text[i_e:]
            results.append(_rec(pert, sev, "adverb-weakening", "intensifier", f"{inten}→{weaker}"))

    return results

def p06_pragmatic(text, sentiment, ridx):
    """P06: Pragmatic Shift — phrase-replacement."""
    pools = PRAGMATIC_POS if sentiment=="positive" else PRAGMATIC_NEG
    results = []
    tl = text.lower()

    for sev in SEVERITY:
        mapping = pools[sev]
        for phrase in sorted(mapping, key=len, reverse=True):
            idx = tl.find(phrase)
            if idx != -1:
                repl = mapping[phrase]
                pert = text[:idx] + repl + text[idx+len(phrase):]
                results.append(_rec(pert, sev, "phrase-replacement", "clause", f"{phrase}→{repl}"))
                break

    return results

def p07_contrastive(text, sentiment, ridx):
    """P07: Contrastive Neutralisation — adjective-replacement."""
    pools = CONTRAST_POS if sentiment=="positive" else CONTRAST_NEG
    adj_s, adj_e, adj_w = _find_predicate_adj(text)
    results = []

    for sev in SEVERITY:
        mapping = pools[sev]
        target = adj_w.lower() if adj_w else None
        repl = mapping.get(target) if target else None
        if repl is None: continue
        pert = text[:adj_s] + repl + text[adj_e:]
        results.append(_rec(pert, sev, "adjective-replacement", "adjective", f"{target}→{repl}"))

    return results

def p08_synonym(text, sentiment, ridx):
    """P08: Synonym Replacement — adjective-replacement."""
    pools = SYNONYM_POS if sentiment=="positive" else SYNONYM_NEG
    adj_s, adj_e, adj_w = _find_predicate_adj(text)
    results = []

    for sev in SEVERITY:
        mapping = pools[sev]
        target = adj_w.lower() if adj_w else None
        repl = mapping.get(target) if target else None
        if repl is None: continue
        pert = text[:adj_s] + repl + text[adj_e:]
        results.append(_rec(pert, sev, "adjective-replacement", "adjective", f"{target}→{repl}"))

    return results

def p09_intensity(text, sentiment, ridx):
    """P09: Intensity Scaling — intensifier-removal."""
    i_s, i_e, a_s, a_e, inten, adj_w = _find_intensifier_adj_bigram(text)
    if i_s is None: return []

    pert = re.sub(r'\s+', ' ', text[:i_s] + text[i_e:].lstrip()).strip()
    if pert == text: return []

    return [_rec(pert, "mild", "intensifier-removal", "intensifier", f"removed={inten}")]

def p10_syntax(text, sentiment, ridx):
    """P10: Syntax Rewriting — active-to-passive."""
    results = []

    for sev in SEVERITY:
        for pattern, replacement in SYNTAX_PATTERNS[sev]:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                pert = text[:m.start()] + replacement + text[m.end():]
                results.append(_rec(pert, sev, "active-to-passive", "verb-phrase", f"{m.group()}→{replacement}"))
                break

    return results

def p11_entity(text, sentiment, ridx):
    """P11: Entity Substitution — noun-slot-swap."""
    words = text.split()
    results = []

    for i, word in enumerate(words):
        clean = word.strip(".,!?;:'\"").lower()
        if clean in ENTITY_MAP:
            tiers = ENTITY_MAP[clean]
            punct = re.search(r'[.,!?;:\'\"]+$', word)
            p_str = punct.group() if punct else ""

            for sev, pool in zip(SEVERITY, tiers):
                chosen = _pick(pool, ridx, "P11", sev, "noun-slot-swap")
                if chosen.lower() == clean: continue

                if word[0].isupper(): chosen = chosen[0].upper()+chosen[1:]
                chosen = chosen.rstrip(".,!?;:'\"") + p_str

                nw = list(words)
                nw[i] = chosen
                results.append(_rec(" ".join(nw), sev, "noun-slot-swap", "entity-noun", f"{clean}→{chosen}"))
            break

    return results

def p12_register(text, sentiment, ridx):
    """P12: Cross-Register Rewriting — phrase-replacement."""
    results = []

    for sev in SEVERITY:
        mapping = REGISTER_BY_SEV[sev]
        for slang in sorted(mapping, key=len, reverse=True):
            m = re.search(r'\b' + re.escape(slang) + r'\b', text, re.IGNORECASE)
            if m:
                repl = mapping[slang]
                pert = text[:m.start()] + repl + text[m.end():]
                results.append(_rec(pert, sev, "phrase-replacement", "register-expression", f"{slang}→{repl}"))
                break

    return results

# Family registry
FAMILIES = {
    "P01_hedging": p01_hedging,
    "P02_discourse": p02_discourse,
    "P03_negation": p03_negation,
    "P04_sarcasm": p04_sarcasm,
    "P05_dilution": p05_dilution,
    "P06_pragmatic": p06_pragmatic,
    "P07_contrastive": p07_contrastive,
    "P08_synonym": p08_synonym,
    "P09_intensity": p09_intensity,
    "P10_syntax": p10_syntax,
    "P11_entity": p11_entity,
    "P12_register": p12_register,
}

def apply_all_perturbations(review: str, sentiment: str, review_idx: int) -> list:
    """Apply all 12 families. De-duplicate on perturbed string."""
    sent = sentiment.strip().lower()
    results = []
    seen = set()

    for family, func in FAMILIES.items():
        try:
            candidates = func(review, sent, review_idx)
        except Exception:
            candidates = []

        for item in candidates:
            p = item["perturbed_review"].strip()
            if not p or p == review.strip() or p in seen:
                continue
            seen.add(p)
            item["perturbation_family"] = family
            results.append(item)

    return results

# ═════════════════════════════════════════════════════════════════════════════
# METRIC ENGINE (with fallbacks)
# ═════════════════════════════════════════════════════════════════════════════

class MetricEngine:
    def __init__(self):
        self._sbert = None
        self._nli = None
        self._ppl_m = None
        self._ppl_t = None
        self._vader = None
        self._check()

    def _check(self):
        self.has_sbert = self._probe("sentence_transformers")
        self.has_nli = self._probe("transformers")
        self.has_ppl = self._probe("transformers")
        self.has_vader = self._probe("vaderSentiment")
        self.has_sklearn = self._probe("sklearn")

    @staticmethod
    def _probe(pkg):
        try:
            __import__(pkg.split(".")[0])
            return True
        except:
            return False

    def _load_sbert(self):
        if self._sbert is None and self.has_sbert:
            from sentence_transformers import SentenceTransformer
            print("  [load] SBERT...", flush=True)
            self._sbert = SentenceTransformer("all-MiniLM-L6-v2")
        return self._sbert

    def _load_nli(self):
        if self._nli is None and self.has_nli:
            from transformers import pipeline
            print("  [load] NLI...", flush=True)
            self._nli = pipeline("text-classification",
                                  model="cross-encoder/nli-MiniLM2-L6-H768",
                                  device=-1, top_k=None)
        return self._nli

    def _load_ppl(self):
        if self._ppl_m is None and self.has_ppl:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            print("  [load] distilgpt2...", flush=True)
            self._ppl_t = AutoTokenizer.from_pretrained("distilgpt2")
            self._ppl_m = AutoModelForCausalLM.from_pretrained("distilgpt2")
            self._ppl_m.eval()
        return self._ppl_m, self._ppl_t

    def _load_vader(self):
        if self._vader is None and self.has_vader:
            from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
            self._vader = SentimentIntensityAnalyzer()
        return self._vader

    @staticmethod
    def _tok(t):
        return re.findall(r"\b\w+\b", t.lower())

    def compute(self, orig: str, pert: str) -> dict:
        """Compute all 8 metrics."""
        ot, pt = self._tok(orig), self._tok(pert)

        # M1: SBERT cosine (with TF-IDF fallback)
        try:
            sbert = self._load_sbert()
            if sbert:
                embs = sbert.encode([orig, pert], normalize_embeddings=True)
                m1 = round(float(np.dot(embs[0], embs[1])), 4)
            else:
                m1 = 0.5
        except:
            m1 = 0.5

        # M2: ROUGE-L (pure Python)
        m, n = len(ot), len(pt)
        if m and n:
            prev = [0]*(n+1)
            for i in range(1, m+1):
                curr = [0]*(n+1)
                for j in range(1, n+1):
                    curr[j] = (prev[j-1]+1) if ot[i-1]==pt[j-1] else max(curr[j-1], prev[j])
                prev = curr
            lcs = prev[n]
            m2 = round(2*(lcs/n)*(lcs/m)/((lcs/n)+(lcs/m)), 4) if lcs > 0 else 0.0
        else:
            m2 = 0.0

        # M3: NLI (with overlap fallback)
        try:
            nli = self._load_nli()
            if nli:
                res = nli(orig, text_pair=pert, truncation=True, max_length=512)
                lbs = res if isinstance(res, list) else [res]
                m3_o2p = round(max([x.get("score",0) for x in lbs if "ENTAIL" in x.get("label","").upper()]+[0.0]), 4)
            else:
                m3_o2p = 0.5
        except:
            m3_o2p = 0.5

        try:
            nli = self._load_nli()
            if nli:
                res = nli(pert, text_pair=orig, truncation=True, max_length=512)
                lbs = res if isinstance(res, list) else [res]
                m3_p2o = round(max([x.get("score",0) for x in lbs if "ENTAIL" in x.get("label","").upper()]+[0.0]), 4)
            else:
                m3_p2o = 0.5
        except:
            m3_p2o = 0.5

        # M4: VADER (with lexicon fallback)
        try:
            vader = self._load_vader()
            if vader:
                m4_orig = round(vader.polarity_scores(orig)["compound"], 4)
                m4_pert = round(vader.polarity_scores(pert)["compound"], 4)
            else:
                m4_orig = m4_pert = 0.0
        except:
            m4_orig = m4_pert = 0.0

        # M5: PPL (with entropy fallback)
        try:
            ppl_m, ppl_t = self._load_ppl()
            if ppl_m:
                import torch
                enc = ppl_t(orig, return_tensors="pt", truncation=True, max_length=256)
                ids = enc["input_ids"]
                if ids.shape[1] >= 2:
                    with torch.no_grad():
                        loss = ppl_m(ids, labels=ids).loss
                    m5_orig = round(float(torch.exp(loss).item()), 4)
                else:
                    m5_orig = 50.0

                enc = ppl_t(pert, return_tensors="pt", truncation=True, max_length=256)
                ids = enc["input_ids"]
                if ids.shape[1] >= 2:
                    with torch.no_grad():
                        loss = ppl_m(ids, labels=ids).loss
                    m5_pert = round(float(torch.exp(loss).item()), 4)
                else:
                    m5_pert = 50.0
            else:
                m5_orig = m5_pert = 1.0
        except:
            m5_orig = m5_pert = 1.0

        # M6: Edit distance (pure Python)
        a, b = orig, pert
        if len(a) < len(b): a, b = b, a
        if not b:
            lev = len(a)
        else:
            prev = list(range(len(b)+1))
            for ca in a:
                curr = [prev[0]+1]
                for j, cb in enumerate(b):
                    curr.append(min(curr[j]+1, prev[j+1]+1, prev[j]+(0 if ca==cb else 1)))
                prev = curr
            lev = prev[-1]
        m6 = round(lev / max(len(orig), len(pert), 1), 4)

        # M7: PPC (from M4)
        m7 = round(1.0 - abs(m4_orig - m4_pert) / 2.0, 4)

        # Composite Φ
        _W = {"m1": 0.30, "m3": 0.25, "m4": 0.20, "m2": 0.10, "m5": 0.10, "m6": 0.05}
        phi = round(
            _W["m1"] * (1.0 if m1 >= 0.80 else 0.0) +
            _W["m3"] * (1.0 if m3_o2p >= 0.40 and m3_p2o >= 0.40 else 0.0) +
            _W["m4"] * (1.0 if (m4_orig >= 0.05 and m4_pert >= 0.05) or (m4_orig <= -0.05 and m4_pert <= -0.05) else 0.0) +
            _W["m2"] * (1.0 if m2 >= 0.40 else 0.0) +
            _W["m5"] * (1.0 if m5_pert / max(m5_orig, 0.001) <= 3.0 else 0.0) +
            _W["m6"] * (1.0 if m6 <= 0.60 else 0.0),
            4)

        # M8: PQS
        m8 = round(phi * (1.0 - m6), 4)

        return {
            "m1_cosine_sim": m1,
            "m1_pass": m1 >= 0.80,
            "m2_bleu1": round(sum(1 for w in pt if w in set(ot)) / max(len(pt), 1), 4) if pt else 0.0,
            "m2_rougeL": m2,
            "m2_pass": m2 >= 0.40,
            "m3_nli_o2p": m3_o2p,
            "m3_nli_p2o": m3_p2o,
            "m3_pass": m3_o2p >= 0.40 and m3_p2o >= 0.40,
            "m4_vader_orig": m4_orig,
            "m4_vader_pert": m4_pert,
            "m4_pass": (m4_orig >= 0.05 and m4_pert >= 0.05) or (m4_orig <= -0.05 and m4_pert <= -0.05),
            "m5_ppl_orig": m5_orig,
            "m5_ppl_pert": m5_pert,
            "m5_ppl_ratio": round(m5_pert / max(m5_orig, 0.001), 4),
            "m5_pass": m5_pert / max(m5_orig, 0.001) <= 3.0,
            "m6_edit_dist": m6,
            "m6_pass": m6 <= 0.60,
            "m7_ppc": m7,
            "m7_pass": m7 >= 0.75,
            "sem_preservation_score": phi,
            "sem_preserved": phi >= 0.75,
            "m8_pqs": m8,
        }

# ═════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING
# ═════════════════════════════════════════════════════════════════════════════

OUTPUT_COLS = [
    "review_idx","original_review","perturbed_review","sentiment",
    "perturbation_family","severity","position_strategy","linguistic_target","variant_label","source_dataset",
    "m1_cosine_sim","m1_pass",
    "m2_bleu1","m2_rougeL","m2_pass",
    "m3_nli_o2p","m3_nli_p2o","m3_pass",
    "m4_vader_orig","m4_vader_pert","m4_pass",
    "m5_ppl_orig","m5_ppl_pert","m5_ppl_ratio","m5_pass",
    "m6_edit_dist","m6_pass",
    "m7_ppc","m7_pass",
    "sem_preservation_score","sem_preserved",
    "m8_pqs",
]

def process_dataset(filepath: str, engine: MetricEngine) -> pd.DataFrame:
    try:
        df = pd.read_csv(filepath, encoding="utf-8")
    except:
        df = pd.read_csv(filepath, encoding="latin-1")

    df.columns = [c.strip().lower() for c in df.columns]
    df = df[["review", "sentiment"]].dropna()
    df["review"] = df["review"].astype(str).str.strip()
    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
    df = df[df["sentiment"].isin(["positive", "negative"])].reset_index(drop=True)

    if df.empty:
        return pd.DataFrame()

    rows = []
    for ridx, row in df.iterrows():
        review, sentiment = row["review"], row["sentiment"]
        if len(review.split()) < 2:
            continue

        for item in apply_all_perturbations(review, sentiment, ridx):
            record = {
                "review_idx": ridx,
                "original_review": review,
                "perturbed_review": item["perturbed_review"],
                "sentiment": sentiment,
                "perturbation_family": item["perturbation_family"],
                "severity": item["severity"],
                "position_strategy": item["position_strategy"],
                "linguistic_target": item["linguistic_target"],
                "variant_label": item["variant_label"],
                "source_dataset": "",
            }
            record.update(engine.compute(review, item["perturbed_review"]))
            rows.append(record)

        if (ridx + 1) % 200 == 0:
            print(f"  [{ridx+1:,}] processed, {len(rows):,} variants", flush=True)

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    print(f"  ✓ {len(out):,} rows with all 8 metrics")
    return out

# ═════════════════════════════════════════════════════════════════════════════
# RUN
# ═════════════════════════════════════════════════════════════════════════════

print("Initializing MetricEngine...")
engine = MetricEngine()
print(f"  SBERT: {engine.has_sbert}, NLI: {engine.has_nli}, VADER: {engine.has_vader}\n")

print("▶ PROCESSING DATASETS\n")

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
if not files:
    print(f"✗ No CSV files in {INPUT_DIR}")
else:
    all_dfs = []
    for filepath in files:
        name = os.path.splitext(os.path.basename(filepath))[0]
        print(f"▶  {name}")
        result = process_dataset(filepath, engine)

        if result.empty:
            print(f"  ✗ 0 rows\n")
            continue

        result["source_dataset"] = name
        all_dfs.append(result)

        # Save locally
        local_path = os.path.join(LOCAL_OUTPUT, f"{name}_perturbed.csv")
        result.to_csv(local_path, index=False, encoding="utf-8")
        print(f"  → Saved locally")

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)

        # Save locally
        local_path = os.path.join(LOCAL_OUTPUT, "ALL_DATASETS_perturbed.csv")
        combined.to_csv(local_path, index=False, encoding="utf-8")

        # Save to Google Drive if available
        if SAVE_TO_GDRIVE:
            gd_path = os.path.join(GDRIVE_PATH, "ALL_DATASETS_perturbed.csv")
            combined.to_csv(gd_path, index=False, encoding="utf-8")
            print(f"\n  ✓ Saved to Google Drive: {gd_path}")

        print(f"\n" + "═"*80)
        print(f"✓ PHASE 1 COMPLETE")
        print(f"  Total pairs        : {len(combined):,}")
        print(f"  Families applied   : {combined['perturbation_family'].nunique()}")
        print(f"  Unique positions   : {combined['position_strategy'].nunique()}")
        print(f"  Severity levels    : {combined['severity'].nunique()}")
        print(f"  All 8 metrics      : ✓ (M1-M8 including M7 PPC, M8 PQS)")
        print("═"*80 + "\n")

        print("Families included:")
        for fam in sorted(combined['perturbation_family'].unique()):
            n = len(combined[combined['perturbation_family'] == fam])
            print(f"  {fam}: {n:,}")

        print("\nPositions included:")
        for pos in sorted(combined['position_strategy'].unique()):
            n = len(combined[combined['position_strategy'] == pos])
            print(f"  {pos}: {n:,}")

        print("\nSeverity distribution:")
        for sev in SEVERITY:
            n = len(combined[combined['severity'] == sev])
            print(f"  {sev}: {n:,}")
    else:
        print("\n✗ No data generated")

print("\n✓ Done.\n")


╔══════════════════════════════════════════════════════════════════════════════╗
║                    CSPF PHASE 1 — COMPLETE IMPLEMENTATION                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
✓ Output path: /content/drive/MyDrive/postdoc-papers/paper2/generated files phase1

Initializing MetricEngine...
  SBERT: True, NLI: True, VADER: False

▶ PROCESSING DATASETS

▶  STS-gold
  [load] SBERT...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (22 outputs left out)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

  [200] processed, 2,641 variants
  [400] processed, 5,526 variants


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  CSPF — Phase 2: Transformer Inference  [FINAL MERGED]                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Models: RoBERTa, XLNet, ELECTRA, DistilBERT, ALBERT, BERT, GPT2           ║
║          → standard seq_cls inference                                       ║
║  T5      → fixed: prefix "sentiment: ", output 'p'/'n'                     ║
║  TweeBERTa → fixed: TensorFlow weights, from_tf=True                       ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — HuggingFace token (optional)
# ═══════════════════════════════════════════════════════════════════════════════
import os
HF_TOKEN = ""   # ← paste your token here:  "hf_xxxxxxxxxxxx"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✓ HuggingFace token set.")
else:
    print("⚠ No HF token — may hit rate limits. Get one free at huggingface.co/settings/tokens")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess
subprocess.run(["pip", "install", "-q",
                "transformers", "torch", "tensorflow",
                "sentencepiece", "accelerate",
                "pandas", "numpy", "tqdm"], check=True)
print("✓ Dependencies installed")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Config
# ═══════════════════════════════════════════════════════════════════════════════
import gc, zipfile, warnings
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM,
                          T5ForConditionalGeneration,
                          T5Tokenizer)

warnings.filterwarnings("ignore")

INPUT_CSV  = "/content/drive/MyDrive/1/postdoc-papers/paper2/cspf_output/ALL_DATASETS_perturbed.csv"
# اگر مستقیم در Colab آپلود کردی:
# INPUT_CSV = "/content/ALL_DATASETS_perturbed.csv"
OUTPUT_DIR = "cspf_inference_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32 if DEVICE == "cuda" else 8
MAX_LENGTH = 128

print(f"Device     : {DEVICE}")
if DEVICE == "cpu":
    print("  ⚠ WARNING: CPU mode will be very slow.")
    print("    → Runtime → Change runtime type → T4 GPU")
print(f"Batch size : {BATCH_SIZE}")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Model registry
# ═══════════════════════════════════════════════════════════════════════════════
MODEL_REGISTRY = [
    ("RoBERTa",    "Etelis/Sentiment140_roBERTa_5E",                    "seq_cls"),
    ("XLNet",      "Etelis/Sentiment140_XLNET_5E",                      "seq_cls"),
    ("ELECTRA",    "Etelis/Sentiment140_ELECTRA_5E",                    "seq_cls"),
    ("DistilBERT", "Etelis/Sentiment140_DistilBERT_5E",                 "seq_cls"),
    ("ALBERT",     "Etelis/Sentiment140_ALBERT_5E",                     "seq_cls"),
    ("BERT",       "Etelis/Sentiment140_BERT_5E",                       "seq_cls"),
    ("GPT2",       "ashok2216/gpt2-amazon-sentiment-classifier-V1.0",   "seq_cls"),
    ("TweeBERTa",  "adityapatkar/TweeBERTa",                            "seq_cls_tf"),
    ("T5",         "michelecafagna26/t5-base-finetuned-sst2-sentiment", "seq2seq_t5"),
]


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Label helpers
# ═══════════════════════════════════════════════════════════════════════════════

def normalise_seq_cls(raw: str) -> str:
    """Standard label normaliser for seq_cls models."""
    s = str(raw).strip().lower()
    lmap = {
        "positive": "positive", "negative": "negative",
        "pos":      "positive", "neg":      "negative",
        "1":        "positive", "0":        "negative",
        "label_1":  "positive", "label_0":  "negative",
        "label_2":  "positive",
        "4":        "positive",
        "positive sentiment": "positive",
        "negative sentiment": "negative",
        "positive.": "positive", "negative.": "negative",
    }
    if s in lmap:
        return lmap[s]
    if "pos" in s: return "positive"
    if "neg" in s: return "negative"
    try:
        return "positive" if int(s) >= 1 else "negative"
    except ValueError:
        pass
    return "positive"


def decode_t5(raw: str) -> str:
    """
    T5-specific decoder (michelecafagna26/t5-base-finetuned-sst2-sentiment).

    Per HF model card:
      prefix = "sentiment: <text>"
      output = 'p' (positive) or 'n' (negative)

    Fallbacks handle edge cases where full words appear.
    """
    s = str(raw).strip().lower()
    if not s:
        return "positive"
    # Primary: single-char output (documented model behavior)
    if s == "p": return "positive"
    if s == "n": return "negative"
    # Fallback: substring search
    if "neg" in s: return "negative"
    if "pos" in s: return "positive"
    # Numeric
    if s in ("0", "false"): return "negative"
    if s in ("1", "true"):  return "positive"
    return "positive"


def get_id2label(model) -> dict:
    cfg = model.config
    if hasattr(cfg, "id2label") and cfg.id2label:
        return {int(k): str(v) for k, v in cfg.id2label.items()}
    return {0: "negative", 1: "positive"}


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Model loading helpers
# ═══════════════════════════════════════════════════════════════════════════════

def load_seq_cls_safe(model_id: str, device: str):
    """
    Load a seq_cls model trying multiple strategies.
    Loads in float32, then casts to float16 on GPU.
    """
    strategies = [
        lambda: AutoModelForSequenceClassification.from_pretrained(model_id),
        lambda: AutoModelForSequenceClassification.from_pretrained(
                    model_id, trust_remote_code=True),
        lambda: AutoModelForSequenceClassification.from_pretrained(
                    model_id, ignore_mismatched_sizes=True),
    ]
    last_err = None
    for i, strategy in enumerate(strategies, 1):
        try:
            model = strategy().to(device)
            if device == "cuda":
                model = model.half()
            print(f"  Model loaded (strategy {i})")
            return model
        except Exception as e:
            last_err = e
            if i < len(strategies):
                print(f"  Strategy {i} failed: {type(e).__name__}: {str(e)[:80]}")
                print(f"  Trying strategy {i+1} …")
    raise RuntimeError(f"All loading strategies failed. Last error: {last_err}")


def load_tweeberta(model_id: str, device: str):
    """
    TweeBERTa uses TensorFlow weights — must use from_tf=True.
    """
    try:
        import tensorflow as tf  # noqa: F401 — required for TF weight loading
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            from_tf=True,
            trust_remote_code=True
        ).to(device)
        if device == "cuda":
            model = model.half()
        print("  ✓ TweeBERTa loaded (TF → PT)")
        return model
    except Exception as e:
        print(f"  ✗ TweeBERTa loading failed: {e}")
        print("  Fix: pip install tensorflow >= 2.13")
        return None


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Inference functions
# ═══════════════════════════════════════════════════════════════════════════════

def predict_seq_cls(texts: list, tokenizer, model, id2label: dict,
                    batch_size: int = BATCH_SIZE,
                    desc: str = "Predicting") -> list:
    """Standard seq_cls inference with float16/32 mismatch handling."""
    model.eval()
    model_dtype = next(model.parameters()).dtype
    results = []
    for i in tqdm(range(0, len(texts), batch_size),
                  total=(len(texts) + batch_size - 1) // batch_size,
                  desc=desc, unit="batch", leave=True):
        batch = texts[i: i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            try:
                logits = model(**enc).logits
            except RuntimeError as e:
                if "Half" in str(e) or "Float" in str(e) or "dtype" in str(e):
                    enc = {k: v.to(model_dtype) if v.is_floating_point() else v
                           for k, v in enc.items()}
                    logits = model(**enc).logits
                else:
                    raise
        for p in logits.argmax(dim=-1).cpu().tolist():
            results.append(normalise_seq_cls(id2label.get(p, str(p))))
    return results


def predict_t5(texts: list, tokenizer, model,
               batch_size: int = BATCH_SIZE,
               desc: str = "T5 Predicting") -> list:
    """
    T5 inference with correct prefix "sentiment: " per model card.
    Output is 'p' or 'n' → decoded via decode_t5().
    """
    model.eval()
    results = []
    first = True
    for i in tqdm(range(0, len(texts), batch_size),
                  total=(len(texts) + batch_size - 1) // batch_size,
                  desc=desc, unit="batch", leave=True):
        batch = texts[i: i + batch_size]
        prefixed = ["sentiment: " + t for t in batch]   # ← required prefix
        enc = tokenizer(prefixed, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out_ids = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=3,   # single char 'p' or 'n'
            )
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)

        if first:
            print(f"\n  [T5 DEBUG] First 5 raw outputs:")
            for raw in decoded[:5]:
                print(f"    '{raw}'  →  '{decode_t5(raw)}'")
            first = False

        results.extend(decode_t5(d) for d in decoded)
    return results


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Load CSV
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading {INPUT_CSV} …")
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"  {len(df):,} rows  |  {df.shape[1]} columns")

for col in ["review_idx", "original_review", "perturbed_review", "sentiment"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: '{col}'")

unique_orig = (df[["review_idx", "original_review"]]
               .drop_duplicates(subset="review_idx")
               .reset_index(drop=True))
unique_orig_texts = unique_orig["original_review"].tolist()
unique_orig_idx   = unique_orig["review_idx"].tolist()
perturbed_texts   = df["perturbed_review"].tolist()

print(f"  Unique originals : {len(unique_orig_texts):,}")
print(f"  Perturbed rows   : {len(df):,}\n")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Main inference loop
# ═══════════════════════════════════════════════════════════════════════════════

all_model_names = []

for (model_name, model_id, model_class) in MODEL_REGISTRY:

    orig_col = f"{model_name}_orig_label"
    pert_col = f"{model_name}_pert_label"
    flip_col = f"{model_name}_flip"

    print("=" * 70)
    print(f"  Model : {model_name}  ({model_id})")
    print("=" * 70)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    ckpt_path = os.path.join(OUTPUT_DIR, f"{model_name}_predictions.csv")
    if os.path.exists(ckpt_path):
        try:
            cached = pd.read_csv(ckpt_path, low_memory=False)
            if all(c in cached.columns for c in [orig_col, pert_col, flip_col]):
                print(f"  ✓ Checkpoint found — loading cache, skipping inference.")
                idx_to_orig = (cached.set_index("review_idx")[orig_col].to_dict()
                               if "review_idx" in cached.columns else {})
                if idx_to_orig:
                    df[orig_col] = df["review_idx"].map(idx_to_orig)
                    df[pert_col] = cached[pert_col].values[:len(df)]
                    df[flip_col] = cached[flip_col].values[:len(df)]
                else:
                    df[orig_col] = cached[orig_col].values[:len(df)]
                    df[pert_col] = cached[pert_col].values[:len(df)]
                    df[flip_col] = cached[flip_col].values[:len(df)]
                fr = df[flip_col].mean()
                print(f"  Flip rate (cache): {fr:.4f} ({100*fr:.2f}%)")
                all_model_names.append(model_name)
                print()
                continue
        except Exception as e:
            print(f"  ⚠ Checkpoint unreadable ({e}) — re-running.")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    print(f"  Loading tokenizer …", flush=True)
    try:
        if model_class == "seq2seq_t5":
            tokenizer = T5Tokenizer.from_pretrained(model_id)
        else:
            tokenizer = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print(f"  ✗ Tokenizer failed: {e}. Skipping.")
        continue

    # ── Model ─────────────────────────────────────────────────────────────────
    print(f"  Loading model …", flush=True)
    model = None
    try:
        if model_class == "seq_cls":
            model = load_seq_cls_safe(model_id, DEVICE)
        elif model_class == "seq_cls_tf":
            model = load_tweeberta(model_id, DEVICE)
        elif model_class == "seq2seq_t5":
            model = T5ForConditionalGeneration.from_pretrained(model_id).to(DEVICE)
            if DEVICE == "cuda":
                model = model.half()
            print("  Model loaded")
    except Exception as e:
        print(f"  ✗ Model failed: {e}. Skipping.")
        continue

    if model is None:
        print(f"  ✗ Model is None. Skipping.")
        continue

    # GPT-2 has no pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        if model_class == "seq_cls":
            model.config.pad_token_id = tokenizer.eos_token_id

    id2label = get_id2label(model) if model_class in ("seq_cls", "seq_cls_tf") else {}

    # ── Predict originals ─────────────────────────────────────────────────────
    print(f"  Predicting originals ({len(unique_orig_texts):,} unique) …", flush=True)
    try:
        if model_class == "seq2seq_t5":
            orig_labels = predict_t5(
                unique_orig_texts, tokenizer, model,
                desc=f"{model_name} originals")
        else:
            orig_labels = predict_seq_cls(
                unique_orig_texts, tokenizer, model, id2label,
                desc=f"{model_name} originals")
    except Exception as e:
        print(f"  ✗ Original inference failed: {e}. Skipping.")
        continue

    df[orig_col] = df["review_idx"].map(dict(zip(unique_orig_idx, orig_labels)))

    # ── Predict perturbed ─────────────────────────────────────────────────────
    print(f"  Predicting perturbed ({len(perturbed_texts):,} rows) …", flush=True)
    try:
        if model_class == "seq2seq_t5":
            pert_labels = predict_t5(
                perturbed_texts, tokenizer, model,
                desc=f"{model_name} perturbed")
        else:
            pert_labels = predict_seq_cls(
                perturbed_texts, tokenizer, model, id2label,
                desc=f"{model_name} perturbed")
    except Exception as e:
        print(f"  ✗ Perturbed inference failed: {e}. Skipping.")
        continue

    df[pert_col] = pert_labels
    df[flip_col] = (df[orig_col] != df[pert_col]).astype(int)

    fr = df[flip_col].mean()
    print(f"  ✓ Done. Flip rate: {fr:.4f} ({100*fr:.2f}%)")

    vc = pd.Series(pert_labels).value_counts()
    print(f"  Label distribution (perturbed): {vc.to_dict()}")
    if len(vc) == 1:
        print(f"  ⚠ WARNING: All predictions are '{vc.index[0]}' — check decoder.")

    # ── Save checkpoint ───────────────────────────────────────────────────────
    base_cols = [c for c in df.columns
                 if not any(c.endswith(s)
                            for s in ["_orig_label", "_pert_label", "_flip"])]
    df[base_cols + [orig_col, pert_col, flip_col]].to_csv(
        ckpt_path, index=False, encoding="utf-8")
    print(f"  Saved → {model_name}_predictions.csv")

    all_model_names.append(model_name)

    del model, tokenizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print()


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Combined output, summary, ZIP, download
# ═══════════════════════════════════════════════════════════════════════════════

combined_path = os.path.join(OUTPUT_DIR, "ALL_MODELS_combined.csv")
df.to_csv(combined_path, index=False, encoding="utf-8")
print(f"Combined CSV saved → ALL_MODELS_combined.csv  ({len(df):,} rows)")

print("\n── Flip Rate Summary ──────────────────────────────────────────────────")
print(f"  {'Model':12s}  {'Overall FR':>12}  {'FR on Correct':>14}  {'Flip Count':>10}  {'Status':>10}")
print(f"  {'─'*12}  {'─'*12}  {'─'*14}  {'─'*10}  {'─'*10}")

for mname in all_model_names:
    fc = f"{mname}_flip"
    oc = f"{mname}_orig_label"
    if fc not in df.columns:
        print(f"  {mname:12s}  {'—':>12}  {'—':>14}  {'—':>10}  MISSING")
        continue
    fr_all = df[fc].mean()
    cm     = (df[oc] == df["sentiment"]) if oc in df.columns else pd.Series(False, index=df.index)
    fr_c   = df.loc[cm, fc].mean() if cm.sum() > 0 else float("nan")
    status = "OK" if fr_all > 0.001 else "⚠ CHECK"
    print(f"  {mname:12s}  {fr_all:>12.4f}  {fr_c:>14.4f}  {int(df[fc].sum()):>10,}  {status:>10}")

if "perturbation_family" in df.columns and all_model_names:
    flip_cols = [f"{m}_flip" for m in all_model_names if f"{m}_flip" in df.columns]
    df["mean_flip"] = df[flip_cols].mean(axis=1)
    print("\n── Flip Rate by Perturbation Family (mean across models) ───────────────")
    for fam, fr in (df.groupby("perturbation_family")["mean_flip"]
                    .mean().sort_values(ascending=False).items()):
        print(f"  {fam:38s}: {fr:.4f}")

if "severity" in df.columns and "mean_flip" in df.columns:
    print("\n── Flip Rate by Severity (mean across models) ──────────────────────────")
    for sev in ["mild", "medium", "strong"]:
        fr = df[df["severity"] == sev]["mean_flip"].mean()
        print(f"  {sev:8s}: {fr:.4f}")

zip_path = "cspf_results_final.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for fname in files:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, OUTPUT_DIR))
print(f"\n✓ ZIP: {zip_path}  ({len(all_model_names)} models)")
import shutil; shutil.copy("cspf_results_final.zip", "/content/drive/MyDrive/1/postdoc-papers/paper2/cspf_output2/cspf_results_final.zip")
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Download started.")
except ImportError:
    print(f"  (Not in Colab — {zip_path} is in the working directory.)")

print("\n✓ COMPLETE")

⚠ No HF token — may hit rate limits. Get one free at huggingface.co/settings/tokens


KeyboardInterrupt: 

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  CSPF — Phase 3: Comprehensive Evaluation Pipeline  (FINAL v2)              ║
║  Google Colab Compatible                                                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  INPUT  : ALL_MODELS_combined.csv  (output of Phase 2)                      ║
║  OUTPUT : cspf_evaluation_results/                                           ║
║    01  flip_rate_matrix.csv          FR(family × severity × model) + 95% CI ║
║    02  model_robustness_scores.csv   RS, RS_c, DR, IRR, acc + Bootstrap CI  ║
║    03  severity_monotonicity.csv     Page's L (correct), Spearman ρ         ║
║    04  position_analysis.csv         FR by position + Kruskal-Wallis        ║
║    05  dataset_vulnerability.csv     DVI per dataset                        ║
║    06a kruskal_wallis.csv            H, df, p per family                    ║
║    06b dunn_posthoc.csv              Bonferroni-corrected pairwise z / p    ║
║    07  pir_tensor.pkl                Full 4D tensor [12×3×3×9]              ║
║    08  effect_sizes.csv              Cohen's h + 95% CI (bootstrap)         ║
║    09  semantic_vs_flip.csv          r_pb + Spearman ρ + 95% CI             ║
║    10  severity_drift.csv            Metric means per (metric, fam, sev)    ║
║    11  linguistic_target_fr.csv      FR per linguistic target type           ║
║    12  cross_dataset_fr.csv          FR heatmap (family × dataset)          ║
║    13  agreement_matrix.csv          Fleiss' κ + 95% CI (bootstrap)         ║
║    14  conditional_fr.csv            FR_correct / FR_incorrect               ║
║    15  summary_report.txt            Full plain-text digest                  ║
║    evaluation_results.zip            All files bundled for download          ║
║                                                                              ║
║  FIX v2: Fleiss kappa corrected — uses COUNT matrix not proportion matrix.  ║
║  Previous version produced κ outside [-1,+1]. Now κ ∈ [-1,+1] guaranteed.  ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess
subprocess.run(["pip","install","-q",
                "pandas","numpy","scipy","scikit-learn",
                "statsmodels","pingouin","openpyxl"], check=True)
print("✓ Dependencies ready")

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports & Config
# ═══════════════════════════════════════════════════════════════════════════════
import os, warnings, zipfile, pickle
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, spearmanr, pointbiserialr
from itertools import combinations

warnings.filterwarnings("ignore")
np.random.seed(42)

INPUT_CSV  = "/content/drive/MyDrive/postdoc-papers/paper2/result of labeling/ALL_MODELS_combined.csv"
OUT_DIR    = "cspf_evaluation_results"
N_BOOT     = 1000       # bootstrap iterations for CIs
ALPHA      = 0.05       # significance level
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_CSV} …")
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"  {len(df):,} rows  |  {df.shape[1]} columns")

MODELS     = sorted({c.replace("_flip","") for c in df.columns if c.endswith("_flip")})
FAMILIES   = sorted(df["perturbation_family"].dropna().unique())
SEVERITIES = ["mild","medium","strong"]
POSITIONS  = sorted(df["position_strategy"].dropna().unique()) if "position_strategy" in df.columns else []
DATASETS   = sorted(df["source_dataset"].dropna().unique()) if "source_dataset" in df.columns else []
GT_COL     = "sentiment"

FLIP_COLS  = [f"{m}_flip" for m in MODELS if f"{m}_flip" in df.columns]
ORIG_COLS  = [f"{m}_orig_label" for m in MODELS]
PERT_COLS  = [f"{m}_pert_label" for m in MODELS]

print(f"  Models ({len(MODELS)}): {MODELS}")
print(f"  Families: {len(FAMILIES)}  Datasets: {len(DATASETS)}  Positions: {len(POSITIONS)}")


# ═══════════════════════════════════════════════════════════════════════════════
# STATISTICAL HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def safe_mean(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.mean()) if len(s) else np.nan

def safe_std(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.std()) if len(s) > 1 else np.nan

def proportion_ci_wilson(n_success, n_total, alpha=ALPHA):
    """
    Wilson score confidence interval for a proportion.
    More accurate than normal approximation, especially for small n or extreme p.
    Returns (lower, upper).
    """
    if n_total == 0: return (np.nan, np.nan)
    z    = stats.norm.ppf(1 - alpha/2)
    p    = n_success / n_total
    denom = 1 + z**2 / n_total
    centre = (p + z**2 / (2*n_total)) / denom
    margin = z * np.sqrt(p*(1-p)/n_total + z**2/(4*n_total**2)) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))

def bootstrap_ci(values, stat_fn=np.mean, n_boot=N_BOOT, alpha=ALPHA):
    """
    Percentile bootstrap CI for any statistic.
    Returns (lower, upper, bootstrap_std).
    """
    values = np.array(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) < 2: return (np.nan, np.nan, np.nan)
    boots = [stat_fn(np.random.choice(values, size=len(values), replace=True))
             for _ in range(n_boot)]
    lo = np.percentile(boots, 100*alpha/2)
    hi = np.percentile(boots, 100*(1-alpha/2))
    return (round(lo,4), round(hi,4), round(np.std(boots),4))

def cohen_h(p1, p2):
    """Cohen's h for two proportions. h = 2·arcsin(√p₁) − 2·arcsin(√p₂)."""
    if np.isnan(p1) or np.isnan(p2): return np.nan
    return float(2*np.arcsin(np.sqrt(np.clip(p1,0,1))) -
                 2*np.arcsin(np.sqrt(np.clip(p2,0,1))))

def pages_L_test(fr_values_per_subject):
    """
    Page's L statistic for ordered alternatives.

    Correct implementation:
      - Each 'subject' is one perturbed review (or one review-model data point).
      - Conditions j ∈ {1,2,3} correspond to {mild, medium, strong}.
      - rᵢⱼ = rank of observation i under condition j (ranked within subjects).
      - L = Σⱼ j · (Σᵢ rᵢⱼ)

    fr_values_per_subject: list/array of shape (n, 3) — each row is
      [FR_at_mild, FR_at_medium, FR_at_strong] for one subject.

    Returns (L, z, p_one_tailed).
    """
    mat = np.array(fr_values_per_subject, dtype=float)
    if mat.ndim == 1:
        mat = mat.reshape(1, -1)
    mat = mat[~np.isnan(mat).any(axis=1)]  # drop rows with NaN
    n, k = mat.shape
    if n == 0 or k != 3: return (np.nan, np.nan, np.nan)

    # Rank within each subject (row)
    ranks = np.array([stats.rankdata(row) for row in mat])   # shape (n, k)
    # L statistic
    L = sum((j+1) * ranks[:, j].sum() for j in range(k))
    # Expected value and variance under H0 (no trend)
    mu    = n * k * (k+1)**2 / 4
    sigma = np.sqrt(n * k**2 * (k+1) * (k**2 - 1) / 144)
    z = (L - mu) / sigma if sigma > 0 else 0.0
    p = float(1 - stats.norm.cdf(z))   # one-tailed (H1: increasing)
    return (float(L), float(z), p)

def dunn_bonferroni(groups: dict):
    """
    Dunn's test with Bonferroni correction.
    groups: {name: array_like}
    Returns DataFrame[group1, group2, z, p_raw, p_bonferroni, significant].
    """
    names  = list(groups.keys())
    arrays = [np.array(v, dtype=float) for v in groups.values()]
    all_data  = np.concatenate(arrays)
    all_ranks = stats.rankdata(all_data)
    N = len(all_data); idx = 0
    ranks_per = []
    for a in arrays:
        ranks_per.append(all_ranks[idx:idx+len(a)]); idx += len(a)
    pairs = list(combinations(range(len(names)), 2))
    k = len(pairs); rows = []
    for i, j in pairs:
        ni, nj = len(arrays[i]), len(arrays[j])
        Ri, Rj = ranks_per[i].mean(), ranks_per[j].mean()
        se = np.sqrt((N*(N+1)/12)*(1/ni + 1/nj))
        z  = (Ri - Rj) / se if se > 0 else 0.0
        p_raw  = float(2*(1 - stats.norm.cdf(abs(z))))
        p_bonf = min(p_raw * k, 1.0)
        rows.append({"group1": names[i], "group2": names[j],
                     "z": round(z,4), "p_raw": round(p_raw,6),
                     "p_bonferroni": round(p_bonf,6),
                     f"significant_{ALPHA}": p_bonf < ALPHA})
    return pd.DataFrame(rows)

def fleiss_kappa_with_ci(flip_matrix, n_boot=N_BOOT, alpha=ALPHA):
    """
    Fleiss' kappa + bootstrap 95% CI.
    flip_matrix : (n_items, n_raters) binary array.

    FIX: uses COUNT matrix C[i,j] = number of raters assigning item i to
    category j. Previous version passed proportions to a count-based formula
    producing kappa outside [-1,+1]. Now correctly builds:
      C[:,0] = n_raters - flip_sum_per_item
      C[:,1] = flip_sum_per_item
    Returns (kappa, z, p, ci_lo, ci_hi).
    """
    def _kappa(count_mat):
        count_mat = np.array(count_mat, dtype=float)
        n, k = count_mat.shape
        N = count_mat.sum(axis=1)
        N_bar = N.mean()
        if N_bar <= 1: return 0.0
        pj = count_mat.sum(axis=0) / (n * N_bar)
        Pe = (pj ** 2).sum()
        Pi = ((count_mat ** 2).sum(axis=1) - N) / (N * (N - 1))
        Pbar = Pi.mean()
        denom = 1 - Pe
        return float((Pbar - Pe) / denom) if denom != 0 else 0.0

    mat = np.array(flip_matrix, dtype=float)
    n_items, n_raters = mat.shape
    n_flip1  = mat.sum(axis=1)
    n_flip0  = n_raters - n_flip1
    count_mat = np.stack([n_flip0, n_flip1], axis=1)

    kappa = _kappa(count_mat)
    pj    = count_mat.sum(axis=0) / (n_items * n_raters)
    Pe    = (pj ** 2).sum()
    se    = np.sqrt(2 / (n_items * n_raters * (n_raters - 1))) if n_items > 1 else 0.0
    z     = kappa / se if se > 0 else 0.0
    p     = float(2 * (1 - stats.norm.cdf(abs(z))))

    boots = []
    for _ in range(n_boot):
        idx = np.random.choice(n_items, size=n_items, replace=True)
        boots.append(_kappa(count_mat[idx]))
    ci_lo = float(np.percentile(boots, 100 * alpha / 2))
    ci_hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return (round(kappa,4), round(z,4), round(p,6), round(ci_lo,4), round(ci_hi,4))


# ═══════════════════════════════════════════════════════════════════════════════
# 01 — FLIP RATE MATRIX  FR(family × severity × model) + 95% Wilson CI
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[01] Flip Rate Matrix + Wilson CIs …")

rows_01 = []
for fam in FAMILIES:
    for sev in SEVERITIES:
        mask = (df["perturbation_family"]==fam) & (df["severity"]==sev)
        sub  = df[mask]
        for model in MODELS:
            fc = f"{model}_flip"
            if fc not in sub.columns: continue
            vals = sub[fc].dropna()
            n_total   = len(vals)
            n_flips   = int(vals.sum())
            fr        = n_flips / n_total if n_total > 0 else np.nan
            ci_lo, ci_hi = proportion_ci_wilson(n_flips, n_total)
            rows_01.append({
                "perturbation_family": fam, "severity": sev, "model": model,
                "n_pairs": n_total, "n_flips": n_flips,
                "flip_rate": round(fr,4) if not np.isnan(fr) else np.nan,
                "CI_lower_95": round(ci_lo,4), "CI_upper_95": round(ci_hi,4),
                "CI_width": round(ci_hi - ci_lo, 4),
            })

fr_df = pd.DataFrame(rows_01)
fr_df.to_csv(f"{OUT_DIR}/01_flip_rate_matrix.csv", index=False)
print(f"  ✓ {len(fr_df):,} rows  (FR range: "
      f"{fr_df['flip_rate'].min():.4f} – {fr_df['flip_rate'].max():.4f})")


# ═══════════════════════════════════════════════════════════════════════════════
# 02 — MODEL ROBUSTNESS SCORES + Bootstrap CI
# RS, RS_conditional, DR, IRR, acc_original
# ═══════════════════════════════════════════════════════════════════════════════
print("[02] Model Robustness Scores + Bootstrap CIs …")

rows_02 = []
for model in MODELS:
    fc = f"{model}_flip"; oc = f"{model}_orig_label"; pc = f"{model}_pert_label"
    if fc not in df.columns: continue

    flip_vals = df[fc].dropna().values
    RS        = float(flip_vals.mean()) if len(flip_vals) else np.nan
    rs_lo, rs_hi, _ = bootstrap_ci(flip_vals)

    # Accuracy on original (deduplicated by review_idx)
    if "review_idx" in df.columns and oc in df.columns and GT_COL in df.columns:
        unique = df.drop_duplicates("review_idx")
        acc    = float((unique[oc] == unique[GT_COL]).mean())
    elif oc in df.columns and GT_COL in df.columns:
        acc = float((df[oc] == df[GT_COL]).mean())
    else:
        acc = np.nan

    # Conditional: only correctly classified originals
    if oc in df.columns and GT_COL in df.columns:
        correct_mask = df[oc] == df[GT_COL]
        RS_c = safe_mean(df.loc[correct_mask, fc])
        rc_lo, rc_hi, _ = bootstrap_ci(df.loc[correct_mask, fc].dropna().values)
    else:
        RS_c = rc_lo = rc_hi = np.nan

    # DR: correct original → wrong perturbed
    if oc in df.columns and pc in df.columns and GT_COL in df.columns:
        dr_mask = (df[oc] == df[GT_COL]) & (df[pc] != df[GT_COL])
        DR      = float(dr_mask.mean())
        dr_lo, dr_hi, _ = bootstrap_ci(dr_mask.astype(float).values)
    else:
        DR = dr_lo = dr_hi = np.nan

    # IRR: wrong original → correct perturbed
    if oc in df.columns and pc in df.columns and GT_COL in df.columns:
        irr_mask = (df[oc] != df[GT_COL]) & (df[pc] == df[GT_COL])
        IRR      = float(irr_mask.mean())
    else:
        IRR = np.nan

    rows_02.append({
        "model": model,
        "RS":              round(RS, 4),
        "RS_CI_lo":        rs_lo, "RS_CI_hi": rs_hi,
        "RS_conditional":  round(RS_c,4) if not np.isnan(RS_c) else np.nan,
        "RS_c_CI_lo":      rc_lo, "RS_c_CI_hi": rc_hi,
        "DR_degradation":  round(DR,4)  if not np.isnan(DR)  else np.nan,
        "DR_CI_lo":        dr_lo, "DR_CI_hi": dr_hi,
        "IRR_improvement": round(IRR,4) if not np.isnan(IRR) else np.nan,
        "acc_original":    round(acc,4) if not np.isnan(acc)  else np.nan,
        "n_total_pairs":   int(df[fc].notna().sum()),
        "n_flips_total":   int(df[fc].sum()),
    })

rs_df = pd.DataFrame(rows_02).sort_values("RS")
rs_df.to_csv(f"{OUT_DIR}/02_model_robustness_scores.csv", index=False)
print("  Robustness ranking (lower RS = more robust):")
for _, r in rs_df.iterrows():
    print(f"    {r['model']:12s}  RS={r['RS']:.4f} [{r['RS_CI_lo']:.4f},{r['RS_CI_hi']:.4f}]  "
          f"DR={r['DR_degradation']:.4f}  acc={r['acc_original']:.4f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 03 — SEVERITY MONOTONICITY  Page's L (correct implementation)
#
# For each (family, model) pair we build the per-review FR matrix:
#   rows = unique reviews that appear in all 3 severity levels for this family
#   cols = [FR_at_mild, FR_at_medium, FR_at_strong]  (binary: 0 or 1 per review)
# Then apply Page's L across these n subjects.
# ═══════════════════════════════════════════════════════════════════════════════
print("[03] Severity Monotonicity — Page's L (per-review subject matrix) …")

rows_03 = []
for fam in FAMILIES:
    sub_fam = df[df["perturbation_family"]==fam]
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in sub_fam.columns: continue

        # Build per-review flip indicator across severity levels
        sev_data = {}
        for sev in SEVERITIES:
            sev_sub = sub_fam[sub_fam["severity"]==sev][["review_idx", fc]].dropna() \
                if "review_idx" in df.columns \
                else sub_fam[sub_fam["severity"]==sev][[fc]].dropna()
            if "review_idx" in df.columns:
                # One value per review (take mean in case of multiple variants)
                sev_data[sev] = sev_sub.groupby("review_idx")[fc].mean()
            else:
                sev_data[sev] = sev_sub[fc].reset_index(drop=True)

        # Aggregate FR per severity as the subject-level matrix
        if "review_idx" in df.columns:
            common_idx = set(sev_data["mild"].index)
            for sev in SEVERITIES[1:]:
                common_idx &= set(sev_data[sev].index)
            common_idx = sorted(common_idx)
            if len(common_idx) < 3:
                # Fallback: use aggregate FR per severity as a single-row matrix
                fr_vec = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
                subjects_matrix = [fr_vec]
            else:
                subjects_matrix = [[sev_data[s].loc[i] for s in SEVERITIES]
                                   for i in common_idx]
        else:
            fr_vec = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
            subjects_matrix = [fr_vec]

        L, z_page, p_page = pages_L_test(subjects_matrix)

        fr_vals = [safe_mean(sub_fam[sub_fam["severity"]==s][fc]) for s in SEVERITIES]
        rho, p_spearman = spearmanr([1,2,3], fr_vals) if not any(np.isnan(fr_vals)) else (np.nan, np.nan)

        rows_03.append({
            "perturbation_family": fam, "model": model,
            "FR_mild":   round(fr_vals[0],4) if not np.isnan(fr_vals[0]) else np.nan,
            "FR_medium": round(fr_vals[1],4) if not np.isnan(fr_vals[1]) else np.nan,
            "FR_strong": round(fr_vals[2],4) if not np.isnan(fr_vals[2]) else np.nan,
            "monotone_FR": (fr_vals[0]<=fr_vals[1]<=fr_vals[2])
                            if not any(np.isnan(fr_vals)) else None,
            "n_subjects":  len(subjects_matrix),
            "pages_L":     round(L,2)        if not np.isnan(L)         else np.nan,
            "pages_z":     round(z_page,4)   if not np.isnan(z_page)    else np.nan,
            "pages_p":     round(p_page,6)   if not np.isnan(p_page)    else np.nan,
            "pages_sig":   (p_page < ALPHA)  if not np.isnan(p_page)    else None,
            "spearman_rho":round(rho,4)      if not np.isnan(rho)       else np.nan,
            "spearman_p":  round(p_spearman,6) if not np.isnan(p_spearman) else np.nan,
        })

mono_df = pd.DataFrame(rows_03)
mono_df.to_csv(f"{OUT_DIR}/03_severity_monotonicity.csv", index=False)
n_mono = mono_df["monotone_FR"].sum() if "monotone_FR" in mono_df else 0
n_sig  = mono_df["pages_sig"].sum()   if "pages_sig"  in mono_df else 0
print(f"  Monotone pairs  : {n_mono}/{len(mono_df)}")
print(f"  Page's L sig    : {n_sig}/{len(mono_df)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 04 — POSITION STRATEGY ANALYSIS + Kruskal-Wallis across positions
# ═══════════════════════════════════════════════════════════════════════════════
print("[04] Position Strategy Analysis …")

rows_04 = []
for model in MODELS:
    fc = f"{model}_flip"
    if fc not in df.columns: continue
    groups_pos = {}
    for pos in POSITIONS:
        sub = df[df["position_strategy"]==pos]
        vals = sub[fc].dropna().values
        if len(vals) > 0:
            fr = float(vals.mean())
            n  = len(vals); n_flip = int(vals.sum())
            ci_lo, ci_hi = proportion_ci_wilson(n_flip, n)
            rows_04.append({
                "model": model, "position_strategy": pos,
                "n_pairs": n, "n_flips": n_flip,
                "flip_rate": round(fr,4),
                "CI_lower_95": round(ci_lo,4), "CI_upper_95": round(ci_hi,4),
            })
            groups_pos[pos] = vals
    # Kruskal-Wallis across positions for this model
    if len(groups_pos) >= 2:
        H, p = kruskal(*groups_pos.values())
        rows_04.append({
            "model": model, "position_strategy": "__Kruskal_H__",
            "n_pairs": sum(len(v) for v in groups_pos.values()),
            "n_flips": np.nan, "flip_rate": round(H,4),
            "CI_lower_95": round(p,6), "CI_upper_95": len(groups_pos)-1,
        })

pos_df = pd.DataFrame(rows_04)
pos_df.to_csv(f"{OUT_DIR}/04_position_analysis.csv", index=False)
print(f"  ✓ {len(POSITIONS)} positions × {len(MODELS)} models")


# ═══════════════════════════════════════════════════════════════════════════════
# 05 — DATASET VULNERABILITY INDEX  DVI
# ═══════════════════════════════════════════════════════════════════════════════
print("[05] Dataset Vulnerability Index …")

rows_05 = []
for ds in DATASETS:
    sub_ds = df[df["source_dataset"]==ds] if "source_dataset" in df.columns else df
    model_frs = {}
    for model in MODELS:
        fc = f"{model}_flip"
        if fc in sub_ds.columns:
            v = sub_ds[fc].dropna().values
            if len(v): model_frs[model] = float(v.mean())
    dvi = np.mean(list(model_frs.values())) if model_frs else np.nan
    row = {"source_dataset": ds,
           "DVI": round(dvi,4) if not np.isnan(dvi) else np.nan,
           "n_pairs": len(sub_ds)}
    row.update({f"FR_{m}": round(model_frs.get(m, np.nan),4) for m in MODELS})
    rows_05.append(row)

dvi_df = pd.DataFrame(rows_05).sort_values("DVI", ascending=False)
dvi_df.to_csv(f"{OUT_DIR}/05_dataset_vulnerability.csv", index=False)
print(f"  ✓ {len(rows_05)} datasets  |  DVI range: "
      f"{dvi_df['DVI'].min():.4f} – {dvi_df['DVI'].max():.4f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 06 — KRUSKAL-WALLIS + DUNN POST-HOC
# ═══════════════════════════════════════════════════════════════════════════════
print("[06] Kruskal-Wallis + Dunn post-hoc …")

rows_kw = []; dunn_all = []
for fam in FAMILIES:
    sub_f  = df[df["perturbation_family"]==fam]
    groups = {m: sub_f[f"{m}_flip"].dropna().values
              for m in MODELS if f"{m}_flip" in sub_f.columns
              and len(sub_f[f"{m}_flip"].dropna()) > 0}
    if len(groups) < 2: continue
    H, p = kruskal(*groups.values())
    eta2 = (H - len(groups) + 1) / (len(df[df["perturbation_family"]==fam]) - len(groups)) \
           if (len(df[df["perturbation_family"]==fam]) - len(groups)) > 0 else np.nan
    rows_kw.append({
        "perturbation_family": fam, "kruskal_H": round(H,4),
        "df": len(groups)-1, "kruskal_p": round(p,6),
        "eta_squared": round(eta2,4) if not np.isnan(eta2) else np.nan,
        f"significant_{ALPHA}": p < ALPHA,
    })
    dunn_res = dunn_bonferroni(groups)
    dunn_res["perturbation_family"] = fam
    dunn_all.append(dunn_res)

kw_df = pd.DataFrame(rows_kw)
kw_df.to_csv(f"{OUT_DIR}/06a_kruskal_wallis.csv", index=False)
n_sig_kw = kw_df[f"significant_{ALPHA}"].sum() if len(kw_df) else 0
print(f"  Kruskal-Wallis: {n_sig_kw}/{len(kw_df)} families significant")

if dunn_all:
    dunn_df = pd.concat(dunn_all, ignore_index=True)
    dunn_df.to_csv(f"{OUT_DIR}/06b_dunn_posthoc.csv", index=False)
    n_sig_d = dunn_df[f"significant_{ALPHA}"].sum()
    print(f"  Dunn sig pairs (Bonferroni): {n_sig_d}/{len(dunn_df)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 07 — PIR TENSOR  [12 × 3 × 3 × 9]
# ═══════════════════════════════════════════════════════════════════════════════
print("[07] Building PIR tensor …")

n_F,n_S,n_P,n_M = len(FAMILIES),len(SEVERITIES),max(len(POSITIONS),1),len(MODELS)
pir = np.full((n_F,n_S,n_P,n_M), np.nan)

_pos_list = POSITIONS if POSITIONS else ["ALL"]
for fi,fam in enumerate(FAMILIES):
    for si,sev in enumerate(SEVERITIES):
        for pi,pos in enumerate(_pos_list):
            if POSITIONS:
                mask = ((df["perturbation_family"]==fam) &
                        (df["severity"]==sev) &
                        (df["position_strategy"]==pos))
            else:
                mask = (df["perturbation_family"]==fam) & (df["severity"]==sev)
            sub = df[mask]
            if len(sub)==0: continue
            for mi,model in enumerate(MODELS):
                fc = f"{model}_flip"
                if fc in sub.columns:
                    v = sub[fc].dropna()
                    if len(v): pir[fi,si,pi,mi] = float(v.mean())

with open(f"{OUT_DIR}/07_pir_tensor.pkl","wb") as f:
    pickle.dump({"tensor":pir,
                 "dims":{"0_families":FAMILIES,"1_severities":SEVERITIES,
                         "2_positions":_pos_list,"3_models":MODELS}}, f)
print(f"  Tensor shape: {pir.shape}  |  Non-NaN: {(~np.isnan(pir)).sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# 08 — EFFECT SIZES  Cohen's h + Bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[08] Cohen's h effect sizes + Bootstrap CIs …")

rows_08 = []
for fam in FAMILIES:
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in df.columns: continue
        sub_mild   = df[(df["perturbation_family"]==fam)&(df["severity"]=="mild")][fc].dropna()
        sub_strong = df[(df["perturbation_family"]==fam)&(df["severity"]=="strong")][fc].dropna()
        if len(sub_mild)==0 or len(sub_strong)==0: continue

        p_mild   = float(sub_mild.mean())
        p_strong = float(sub_strong.mean())
        h = cohen_h(p_strong, p_mild)

        # Bootstrap CI for Cohen's h
        boots_h = []
        for _ in range(N_BOOT):
            bm = np.random.choice(sub_mild.values,   size=len(sub_mild),   replace=True).mean()
            bs = np.random.choice(sub_strong.values, size=len(sub_strong), replace=True).mean()
            boots_h.append(cohen_h(bs, bm))
        boots_h = [v for v in boots_h if not np.isnan(v)]
        h_lo = float(np.percentile(boots_h, 100*ALPHA/2))   if boots_h else np.nan
        h_hi = float(np.percentile(boots_h, 100*(1-ALPHA/2))) if boots_h else np.nan

        interp = "large" if abs(h)>=0.8 else ("medium" if abs(h)>=0.5 else "small")
        rows_08.append({
            "perturbation_family": fam, "model": model,
            "FR_mild": round(p_mild,4), "FR_strong": round(p_strong,4),
            "cohen_h": round(h,4) if not np.isnan(h) else np.nan,
            "cohen_h_CI_lo": round(h_lo,4), "cohen_h_CI_hi": round(h_hi,4),
            "effect_size": interp,
        })

eff_df = pd.DataFrame(rows_08)
eff_df.to_csv(f"{OUT_DIR}/08_effect_sizes.csv", index=False)
print(f"  large={( eff_df['effect_size']=='large').sum()}  "
      f"medium={( eff_df['effect_size']=='medium').sum()}  "
      f"small={( eff_df['effect_size']=='small').sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# 09 — SEMANTIC PRESERVATION ↔ FLIP  Point-biserial r + Spearman ρ + CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[09] Semantic × flip correlations …")

rows_09 = []
sem_col = "sem_preservation_score"
if sem_col in df.columns:
    for model in MODELS:
        fc = f"{model}_flip"
        if fc not in df.columns: continue
        sub = df[[sem_col, fc]].dropna()
        if len(sub) < 10: continue

        r_pb, p_pb = pointbiserialr(sub[fc].astype(int), sub[sem_col])
        rho, p_rho = spearmanr(sub[sem_col], sub[fc])

        # Bootstrap CI for r_pb
        boots = []
        arr = sub.values
        for _ in range(N_BOOT):
            idx = np.random.choice(len(arr), size=len(arr), replace=True)
            sample = arr[idx]
            try:
                r_b,_ = pointbiserialr(sample[:,1].astype(int), sample[:,0])
                boots.append(r_b)
            except: pass
        boots = [v for v in boots if not np.isnan(v)]
        r_lo = float(np.percentile(boots, 100*ALPHA/2))   if boots else np.nan
        r_hi = float(np.percentile(boots, 100*(1-ALPHA/2))) if boots else np.nan

        rows_09.append({
            "model": model, "n": len(sub),
            "r_pb": round(r_pb,4), "r_pb_CI_lo": round(r_lo,4), "r_pb_CI_hi": round(r_hi,4),
            "p_pb": round(p_pb,6),
            "spearman_rho": round(rho,4), "spearman_p": round(p_rho,6),
            "direction": "expected (higher Φ→fewer flips)" if r_pb<0 else "unexpected",
        })

sem_df = pd.DataFrame(rows_09)
sem_df.to_csv(f"{OUT_DIR}/09_semantic_vs_flip.csv", index=False)
print(f"  ✓ Mean r_pb = {sem_df['r_pb'].mean():.4f}  "
      f"({'negative = expected' if sem_df['r_pb'].mean()<0 else 'positive = unexpected'})")


# ═══════════════════════════════════════════════════════════════════════════════
# 10 — SEMANTIC METRIC DRIFT ACROSS SEVERITY
# ═══════════════════════════════════════════════════════════════════════════════
print("[10] Metric drift across severity …")

rows_10 = []
metric_cols = [c for c in ["m1_cosine_sim","m2_rougeL","m3_nli_o2p","m3_nli_p2o",
                             "m4_srp","m5_ppl_ratio","m6_edit_dist","m7_lsd","m8_pqs"]
               if c in df.columns]
for mc in metric_cols:
    for fam in FAMILIES:
        for sev in SEVERITIES:
            sub = df[(df["perturbation_family"]==fam)&(df["severity"]==sev)][mc].dropna()
            if len(sub)==0: continue
            rows_10.append({
                "metric": mc, "perturbation_family": fam, "severity": sev,
                "n": len(sub), "mean": round(sub.mean(),4), "std": round(sub.std(),4),
                "median": round(sub.median(),4),
                "q25": round(sub.quantile(0.25),4), "q75": round(sub.quantile(0.75),4),
            })

pd.DataFrame(rows_10).to_csv(f"{OUT_DIR}/10_severity_drift.csv", index=False)
print(f"  ✓ {len(rows_10)} metric-family-severity cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 11 — LINGUISTIC TARGET FR
# ═══════════════════════════════════════════════════════════════════════════════
print("[11] Linguistic target analysis …")
rows_11 = []
if "linguistic_target" in df.columns:
    for lt in sorted(df["linguistic_target"].dropna().unique()):
        sub = df[df["linguistic_target"]==lt]
        for model in MODELS:
            fc = f"{model}_flip"
            if fc not in sub.columns: continue
            vals = sub[fc].dropna()
            if len(vals)==0: continue
            n=len(vals); nf=int(vals.sum())
            ci_lo,ci_hi = proportion_ci_wilson(nf,n)
            rows_11.append({"linguistic_target":lt,"model":model,"n":n,"n_flips":nf,
                             "flip_rate":round(float(vals.mean()),4),
                             "CI_lower_95":round(ci_lo,4),"CI_upper_95":round(ci_hi,4)})
    pd.DataFrame(rows_11).to_csv(f"{OUT_DIR}/11_linguistic_target_fr.csv", index=False)
    print(f"  ✓ {df['linguistic_target'].nunique()} targets")


# ═══════════════════════════════════════════════════════════════════════════════
# 12 — CROSS-DATASET × FAMILY FR HEATMAP DATA
# ═══════════════════════════════════════════════════════════════════════════════
print("[12] Cross-dataset FR heatmap …")
rows_12 = []
if DATASETS:
    for ds in DATASETS:
        sub_ds = df[df["source_dataset"]==ds]
        for fam in FAMILIES:
            sub = sub_ds[sub_ds["perturbation_family"]==fam]
            vals = [safe_mean(sub[f"{m}_flip"]) for m in MODELS
                    if f"{m}_flip" in sub.columns]
            vals = [v for v in vals if not np.isnan(v)]
            mfr = round(np.mean(vals),4) if vals else np.nan
            rows_12.append({"source_dataset":ds,"perturbation_family":fam,
                             "mean_FR_all_models":mfr,"n":len(sub)})
    pd.DataFrame(rows_12).to_csv(f"{OUT_DIR}/12_cross_dataset_fr.csv", index=False)
    print(f"  ✓ {len(rows_12)} dataset-family cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 13 — INTER-MODEL AGREEMENT  Fleiss' κ + Bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
print("[13] Inter-model agreement (Fleiss' κ) + Bootstrap CI …")

rows_13 = []
flip_avail = [f"{m}_flip" for m in MODELS if f"{m}_flip" in df.columns]

if len(flip_avail) >= 2:
    flip_mat = df[flip_avail].fillna(0).astype(int).values  # (n_items, n_raters)

    # Overall
    k,z_k,p_k,ci_lo,ci_hi = fleiss_kappa_with_ci(flip_mat)
    interp = ("almost perfect" if k>0.80 else "substantial" if k>0.60
              else "moderate" if k>0.40 else "fair" if k>0.20 else "poor")
    rows_13.append({"scope":"overall","perturbation_family":"ALL",
                    "fleiss_kappa":k,"z":z_k,"p":p_k,
                    "CI_lo_95":ci_lo,"CI_hi_95":ci_hi,
                    "n_raters":flip_mat.shape[1],"n_items":flip_mat.shape[0],
                    "interpretation":interp})

    # Per family
    for fam in FAMILIES:
        mask  = df["perturbation_family"]==fam
        f_mat = df.loc[mask, flip_avail].fillna(0).astype(int).values
        if f_mat.shape[0] < 5: continue
        k2,z2,p2,l2,h2 = fleiss_kappa_with_ci(f_mat, n_boot=200)
        interp2 = ("almost perfect" if k2>0.80 else "substantial" if k2>0.60
                   else "moderate" if k2>0.40 else "fair" if k2>0.20 else "poor")
        rows_13.append({"scope":"family","perturbation_family":fam,
                        "fleiss_kappa":k2,"z":z2,"p":p2,
                        "CI_lo_95":l2,"CI_hi_95":h2,
                        "n_raters":f_mat.shape[1],"n_items":f_mat.shape[0],
                        "interpretation":interp2})

agree_df = pd.DataFrame(rows_13)
agree_df.to_csv(f"{OUT_DIR}/13_agreement_matrix.csv", index=False)
ov = agree_df[agree_df["scope"]=="overall"]
if len(ov):
    r = ov.iloc[0]
    print(f"  Overall κ = {r['fleiss_kappa']:.4f}  "
          f"[{r['CI_lo_95']:.4f}, {r['CI_hi_95']:.4f}]  ({r['interpretation']})")


# ═══════════════════════════════════════════════════════════════════════════════
# 14 — CONDITIONAL FLIP RATE  FR_correct vs FR_incorrect
# ═══════════════════════════════════════════════════════════════════════════════
print("[14] Conditional Flip Rate …")

rows_14 = []
for model in MODELS:
    fc=f"{model}_flip"; oc=f"{model}_orig_label"
    if fc not in df.columns or oc not in df.columns or GT_COL not in df.columns: continue
    correct = df[oc]==df[GT_COL]
    for fam in FAMILIES:
        fam_mask = df["perturbation_family"]==fam
        for cond_lbl, mask in [("correct_original", correct & fam_mask),
                                ("incorrect_original", ~correct & fam_mask)]:
            sub = df.loc[mask, fc].dropna()
            n=len(sub); nf=int(sub.sum())
            ci_lo,ci_hi = proportion_ci_wilson(nf,n)
            rows_14.append({
                "model":model,"perturbation_family":fam,"condition":cond_lbl,
                "n":n,"n_flips":nf,
                "flip_rate":round(float(sub.mean()),4) if n>0 else np.nan,
                "CI_lower_95":round(ci_lo,4),"CI_upper_95":round(ci_hi,4),
            })

pd.DataFrame(rows_14).to_csv(f"{OUT_DIR}/14_conditional_fr.csv", index=False)
print(f"  ✓ {len(rows_14)} model-family-condition cells")


# ═══════════════════════════════════════════════════════════════════════════════
# 15 — SUMMARY REPORT
# ═══════════════════════════════════════════════════════════════════════════════
print("[15] Writing summary report …")

SEP  = "═"*72
sep2 = "─"*72

lines = [SEP,
"  CSPF PHASE-3 EVALUATION — COMPLETE SUMMARY REPORT",
f"  {len(df):,} pairs  |  {len(MODELS)} models  |  {len(FAMILIES)} families  |  "
f"n_boot={N_BOOT}  α={ALPHA}",
SEP,""]

# Robustness ranking
lines += ["1. MODEL ROBUSTNESS RANKING  (RS ± 95% Bootstrap CI)", sep2]
for _,r in rs_df.iterrows():
    lines.append(f"  {r['model']:12s}  RS={r['RS']:.4f} [{r['RS_CI_lo']:.4f},{r['RS_CI_hi']:.4f}]  "
                 f"DR={r['DR_degradation']:.4f} [{r['DR_CI_lo']:.4f},{r['DR_CI_hi']:.4f}]  "
                 f"acc={r['acc_original']:.4f}")

# FR by family
lines += ["","2. FLIP RATE BY FAMILY (mean across models × severities)", sep2]
fam_fr = fr_df.groupby("perturbation_family")["flip_rate"].mean().sort_values(ascending=False)
for fam,fr in fam_fr.items():
    lines.append(f"  {fam:38s}  FR={fr:.4f}")

# Monotonicity
lines += ["","3. SEVERITY MONOTONICITY (Page's L)", sep2]
if len(mono_df):
    n_valid = mono_df["monotone_FR"].notna().sum()
    pct = 100*mono_df["monotone_FR"].sum()/max(n_valid,1)
    n_sig_p = mono_df["pages_sig"].sum()
    lines += [f"  Monotone pairs       : {int(mono_df['monotone_FR'].sum())}/{n_valid}  ({pct:.1f}%)",
              f"  Page's L significant : {n_sig_p}/{len(mono_df)}  (p < {ALPHA})"]

# Kruskal-Wallis
lines += ["","4. KRUSKAL-WALLIS (cross-model FR per family)", sep2]
if len(kw_df):
    lines.append(f"  Significant: {n_sig_kw}/{len(kw_df)} families  (α = {ALPHA}, Bonf-corrected pairwise)")

# Effect sizes
lines += ["","5. EFFECT SIZES (Cohen's h: FR_strong vs FR_mild)", sep2]
if len(eff_df):
    for cat in ["large","medium","small"]:
        n = (eff_df["effect_size"]==cat).sum()
        lines.append(f"  {cat:6s}: {n:3d} family-model pairs")

# Fleiss kappa
lines += ["","6. INTER-MODEL AGREEMENT (Fleiss' κ)", sep2]
if len(agree_df):
    r = agree_df[agree_df["scope"]=="overall"].iloc[0]
    lines.append(f"  κ = {r['fleiss_kappa']:.4f}  95% CI [{r['CI_lo_95']:.4f}, {r['CI_hi_95']:.4f}]  "
                 f"({r['interpretation']})")

# Semantic correlation
lines += ["","7. SEMANTIC PRESERVATION ↔ FLIP CORRELATION (r_pb + 95% CI)", sep2]
if len(sem_df):
    for _,r in sem_df.iterrows():
        lines.append(f"  {r['model']:12s}  r_pb={r['r_pb']:.4f} [{r['r_pb_CI_lo']:.4f},{r['r_pb_CI_hi']:.4f}]  "
                     f"ρ={r['spearman_rho']:.4f}")

# DVI
lines += ["","8. DATASET VULNERABILITY (top 5)", sep2]
if len(dvi_df):
    for _,r in dvi_df.head(5).iterrows():
        lines.append(f"  {str(r['source_dataset']):30s}  DVI={r['DVI']:.4f}  n={r['n_pairs']:,}")

lines += ["", SEP,
"  FILES PRODUCED",sep2,
"  01  flip_rate_matrix.csv          FR(family×severity×model) + Wilson 95% CI",
"  02  model_robustness_scores.csv   RS, RS_c, DR, IRR, acc + Bootstrap 95% CI",
"  03  severity_monotonicity.csv     Page's L (per-review), Spearman ρ",
"  04  position_analysis.csv         FR by position + Kruskal-Wallis",
"  05  dataset_vulnerability.csv     DVI + per-model FR per dataset",
"  06a kruskal_wallis.csv            H, df, p, η² per family",
"  06b dunn_posthoc.csv              Dunn z / p_bonferroni per pair",
"  07  pir_tensor.pkl                4D PIR tensor [F×S×P×M]",
"  08  effect_sizes.csv              Cohen's h + Bootstrap 95% CI",
"  09  semantic_vs_flip.csv          r_pb + Bootstrap CI + Spearman ρ",
"  10  severity_drift.csv            Metric means/std/Q1/Q3 per (metric,fam,sev)",
"  11  linguistic_target_fr.csv      FR per linguistic target + CI",
"  12  cross_dataset_fr.csv          FR heatmap (family × dataset)",
"  13  agreement_matrix.csv          Fleiss' κ + Bootstrap 95% CI overall+family",
"  14  conditional_fr.csv            FR_correct / FR_incorrect per (model,family)+CI",
"  15  summary_report.txt            This file",
SEP,""]

report = "\n".join(lines)
with open(f"{OUT_DIR}/15_summary_report.txt","w") as f: f.write(report)
print(report)

# ═══════════════════════════════════════════════════════════════════════════════
# ZIP + DOWNLOAD
# ═══════════════════════════════════════════════════════════════════════════════
zip_path = "evaluation_results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for root,_,files in os.walk(OUT_DIR):
        for fname in files:
            fpath = os.path.join(root,fname)
            zf.write(fpath, os.path.relpath(fpath,OUT_DIR))
print(f"\n✓ ZIP: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Download triggered.")
except ImportError:
    print(f"  (Not in Colab — {zip_path} is in the working directory.)")

✓ Dependencies ready
Loading /content/drive/MyDrive/postdoc-papers/paper2/result of labeling/ALL_MODELS_combined.csv …
  807,851 rows  |  56 columns
  Models (8): ['ALBERT', 'BERT', 'DistilBERT', 'ELECTRA', 'GPT2', 'RoBERTa', 'T5', 'XLNet']
  Families: 12  Datasets: 22  Positions: 10

[01] Flip Rate Matrix + Wilson CIs …
  ✓ 288 rows  (FR range: 0.2793 – 0.6923)
[02] Model Robustness Scores + Bootstrap CIs …
  Robustness ranking (lower RS = more robust):
    T5            RS=0.3683 [0.3673,0.3694]  DR=0.0874  acc=0.8435
    RoBERTa       RS=0.3684 [0.3673,0.3694]  DR=0.0925  acc=0.8427
    ELECTRA       RS=0.3714 [0.3703,0.3725]  DR=0.1009  acc=0.8520
    XLNet         RS=0.3725 [0.3714,0.3736]  DR=0.0997  acc=0.8425
    DistilBERT    RS=0.3793 [0.3782,0.3803]  DR=0.1164  acc=0.8082
    GPT2          RS=0.3818 [0.3807,0.3828]  DR=0.1080  acc=0.8183
    BERT          RS=0.3828 [0.3817,0.3838]  DR=0.1217  acc=0.8075
    ALBERT        RS=0.3859 [0.3848,0.3870]  DR=0.1215  acc=0.8012
[03] 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered.


In [ ]:
#plots
"""
CSPF Phase 3 — Advanced Visualization Suite
10 distinct plot types, each unique and publication-ready
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import LineCollection
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, Wedge, FancyBboxPatch
from matplotlib.ticker import MultipleLocator
import seaborn as sns
from scipy.stats import gaussian_kde
import pickle, warnings, os
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
DATA = "/results"
OUT  = "/mnt/user-data/outputs"
os.makedirs(OUT, exist_ok=True)

# ── load all files ─────────────────────────────────────────────────────────────
fr_df   = pd.read_csv(f"{DATA}/01_flip_rate_matrix.csv")
rs_df   = pd.read_csv(f"{DATA}/02_model_robustness_scores.csv")
mono_df = pd.read_csv(f"{DATA}/03_severity_monotonicity.csv")
pos_df  = pd.read_csv(f"{DATA}/04_position_analysis.csv")
dvi_df  = pd.read_csv(f"{DATA}/05_dataset_vulnerability.csv")
kw_df   = pd.read_csv(f"{DATA}/06a_kruskal_wallis.csv")
eff_df  = pd.read_csv(f"{DATA}/08_effect_sizes.csv")
sem_df  = pd.read_csv(f"{DATA}/09_semantic_vs_flip.csv")
drift_df= pd.read_csv(f"{DATA}/10_severity_drift.csv")
ling_df = pd.read_csv(f"{DATA}/11_linguistic_target_fr.csv")
xds_df  = pd.read_csv(f"{DATA}/12_cross_dataset_fr.csv")
kap_df  = pd.read_csv(f"{DATA}/13_agreement_matrix.csv")
cond_df = pd.read_csv(f"{DATA}/14_conditional_fr.csv")
with open(f"{DATA}/07_pir_tensor.pkl","rb") as f: pir_data = pickle.load(f)

# ── global style ───────────────────────────────────────────────────────────────
MODELS   = ['T5','RoBERTa','ELECTRA','XLNet','DistilBERT','GPT2','BERT','ALBERT']
FAMILIES = ['P01_hedging','P02_discourse','P03_negation','P04_sarcasm',
            'P05_dilution','P06_pragmatic','P07_contrastive','P08_synonym',
            'P09_intensity','P10_syntax','P11_entity','P12_register']
FAM_SHORT= ['Hedg','Disc','Neg','Sarc','Dil','Prag','Cont','Syn',
            'Inten','Synt','Ent','Reg']
SEVS     = ['mild','medium','strong']
SEV_SYM  = {'mild':'○','medium':'△','strong':'★'}

# Palette
C_MODELS = ['#1A1A2E','#16213E','#0F3460','#533483','#E94560',
            '#F5A623','#7ED321','#50E3C2']
C_FAM    = plt.cm.tab20(np.linspace(0,1,12))
C_GLOW   = '#00D4FF'

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':150,'savefig.dpi':200,
    'axes.titlesize':12,'axes.labelsize':10,
})

print("Generating 10 advanced plots...")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 1 — Diverging Lollipop: RS vs Accuracy trade-off
#         (not a bar chart — lollipops with dual encoding)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')

rs_sorted = rs_df.sort_values('RS')
rs_vals   = rs_sorted['RS'].values
acc_vals  = rs_sorted['acc_original'].values
dr_vals   = rs_sorted['DR_degradation'].values
models_s  = rs_sorted['model'].values
ci_lo     = rs_sorted['RS_CI_lo'].values
ci_hi     = rs_sorted['RS_CI_hi'].values

baseline  = rs_vals.mean()
y_pos     = np.arange(len(models_s))

for i, (m, rs, acc, dr, lo, hi) in enumerate(
        zip(models_s, rs_vals, acc_vals, dr_vals, ci_lo, ci_hi)):
    delta = rs - baseline
    color = '#2ECC71' if delta < 0 else '#E74C3C'
    # Stem
    ax.plot([baseline, rs], [i, i], color=color, lw=2.5, alpha=0.7,
            solid_capstyle='round')
    # CI bracket
    ax.plot([lo, hi], [i, i], color=color, lw=6, alpha=0.25,
            solid_capstyle='round')
    # RS dot (size = accuracy)
    s = 100 + acc * 600
    ax.scatter(rs, i, s=s, color=color, zorder=5, edgecolors='white', lw=1.5)
    # DR annotation dot (inner)
    ax.scatter(rs, i, s=dr*800, color='white', zorder=6, alpha=0.9)
    # Label
    ax.text(rs + 0.0008, i, f'{m}\n(acc={acc:.3f})', va='center', ha='left',
            fontsize=8.5, color='#2C3E50', fontweight='bold')

ax.axvline(baseline, color='#7F8C8D', lw=1.5, ls='--', alpha=0.6, label=f'Mean RS = {baseline:.4f}')
ax.set_yticks(y_pos)
ax.set_yticklabels([''] * len(y_pos))
ax.set_xlabel('Robustness Score (RS) — lower is better', fontsize=11)
ax.set_title('Model Robustness Landscape\n'
             'Dot size ∝ Original Accuracy · Inner white dot size ∝ Degradation Rate · '
             'Green = below-average RS', fontsize=11, pad=12)
ax.legend(loc='lower right', fontsize=9)

# Annotations
ax.text(0.01, 0.98, 'More robust →', transform=ax.transAxes,
        va='top', ha='left', fontsize=8, color='#27AE60', style='italic')
ax.text(0.99, 0.98, '← Less robust', transform=ax.transAxes,
        va='top', ha='right', fontsize=8, color='#E74C3C', style='italic')

plt.tight_layout()
plt.savefig(f"{OUT}/fig01_diverging_lollipop_RS.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 01 — Diverging Lollipop RS")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 2 — Annotated Heatmap: FR(family × model) collapsed over severity
# ══════════════════════════════════════════════════════════════════════════════
pivot = fr_df.groupby(['perturbation_family','model'])['flip_rate'].mean().unstack()
pivot = pivot.reindex(index=FAMILIES, columns=MODELS)

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#0D1117')

cmap = LinearSegmentedColormap.from_list('cspf',
    ['#0D2137','#1A4A6E','#2980B9','#F39C12','#E74C3C','#8E1B1B'])
im = ax.imshow(pivot.values, cmap=cmap, aspect='auto', vmin=0.27, vmax=0.70)

# Annotate every cell
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.iloc[i,j]
        if np.isnan(v): continue
        text_color = 'white' if v > 0.45 else '#DDD'
        ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                fontsize=7.5, color=text_color, fontweight='bold')

ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels(MODELS, rotation=30, ha='right', color='white', fontsize=9)
ax.set_yticks(range(len(FAMILIES)))
ax.set_yticklabels(FAM_SHORT, color='white', fontsize=9)

# Row and column mean bars on edges
row_means = pivot.mean(axis=1).values
col_means = pivot.mean(axis=0).values

ax_right = ax.inset_axes([1.01, 0, 0.04, 1])
ax_right.barh(range(len(FAMILIES)), row_means, color=[cmap((v-0.27)/(0.70-0.27)) for v in row_means])
ax_right.set_xlim(0.27, 0.70); ax_right.axis('off')
ax_right.set_facecolor('#0D1117')

ax_top = ax.inset_axes([0, 1.01, 1, 0.04])
ax_top.bar(range(len(MODELS)), col_means, color=[cmap((v-0.27)/(0.70-0.27)) for v in col_means])
ax_top.set_xlim(-0.5, len(MODELS)-0.5); ax_top.axis('off')
ax_top.set_facecolor('#0D1117')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.12)
cbar.set_label('Mean Flip Rate', color='white', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.set_title('Polarity Flip Rate Heatmap: Family × Model\n'
             'Side bars = marginal means', color='white', fontsize=12, pad=8)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#333')

plt.tight_layout()
plt.savefig(f"{OUT}/fig02_heatmap_family_model.png", bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("  ✓ Fig 02 — Dark Annotated Heatmap")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Slope / Slopegraph: Severity trend (mild→strong) per family
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#FAFAFA')

mean_sev = fr_df.groupby(['perturbation_family','severity'])['flip_rate'].mean().unstack()
mean_sev = mean_sev.reindex(index=FAMILIES, columns=SEVS)
x_pos = {s: i for i, s in enumerate(SEVS)}

for fi, fam in enumerate(FAMILIES):
    vals = mean_sev.loc[fam]
    x = [x_pos[s] for s in SEVS]
    y = vals.values
    valid = ~np.isnan(y)
    if valid.sum() < 2: continue

    color = C_FAM[fi]
    xi, yi = np.array(x)[valid], y[valid]

    # Draw connecting lines with gradient thickness
    for k in range(len(xi)-1):
        ax.plot([xi[k],xi[k+1]], [yi[k],yi[k+1]],
                color=color, lw=2.5, alpha=0.8, solid_capstyle='round')
        # Arrow indicator at midpoint
        mx, my = (xi[k]+xi[k+1])/2, (yi[k]+yi[k+1])/2
        dy = yi[k+1] - yi[k]
        if abs(dy) > 0.002:
            ax.annotate('', xy=(mx+0.05, my + dy*0.4),
                        xytext=(mx-0.05, my - dy*0.4),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

    # Dots per severity
    for k, (xi_, yi_) in enumerate(zip(xi, yi)):
        ax.scatter(xi_, yi_, s=80, color=color, zorder=5,
                   marker=['o','s','^'][k], edgecolors='white', lw=1)

    # Family label at rightmost valid point
    last_x, last_y = xi[-1], yi[-1]
    ax.text(last_x + 0.04, last_y, FAM_SHORT[fi],
            va='center', ha='left', fontsize=8, color=color, fontweight='bold')

ax.set_xticks([0,1,2])
ax.set_xticklabels(['Mild','Medium','Strong'], fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Flip Rate (across all models)', fontsize=11)
ax.set_title('Severity-Instability Slopegraph\nEach line = one perturbation family, '
             'arrows show direction of change', fontsize=12, pad=10)
ax.set_xlim(-0.3, 2.55)
ax.yaxis.set_major_formatter(lambda x, _: f'{x:.3f}')
ax.grid(axis='y', ls='--', alpha=0.3)

# Add shaded region for expected monotone direction
ax.axhspan(0.44, 0.70, alpha=0.05, color='red', label='High instability zone')
plt.tight_layout()
plt.savefig(f"{OUT}/fig03_slopegraph_severity.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 03 — Slopegraph Severity Trend")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Radar / Spider chart: 12-family FR profile per model
# ══════════════════════════════════════════════════════════════════════════════
mean_fm = fr_df.groupby(['model','perturbation_family'])['flip_rate'].mean().unstack()
mean_fm = mean_fm.reindex(index=MODELS, columns=FAMILIES)

N = len(FAMILIES)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8),
                          subplot_kw=dict(projection='polar'))
fig.patch.set_facecolor('#1A1A2E')

for idx, (model, ax_r) in enumerate(zip(MODELS, axes.flatten())):
    vals = mean_fm.loc[model].fillna(mean_fm.mean()).values.tolist()
    vals += vals[:1]

    color = C_MODELS[idx]
    ax_r.set_facecolor('#0D1117')

    # Reference rings
    for ref in [0.3, 0.4, 0.5, 0.6]:
        ax_r.plot(angles, [ref]*len(angles), '--', color='#444', lw=0.5, alpha=0.5)

    # Fill
    ax_r.fill(angles, vals, alpha=0.25, color=color)
    ax_r.plot(angles, vals, color=color, lw=2.0)

    # Dots at vertices
    ax_r.scatter(angles[:-1], vals[:-1], s=40, color=color,
                 zorder=5, edgecolors='white', lw=0.8)

    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(FAM_SHORT, size=7, color='white')
    ax_r.set_ylim(0.25, 0.70)
    ax_r.set_yticks([0.3, 0.4, 0.5, 0.6])
    ax_r.set_yticklabels(['0.30','0.40','0.50','0.60'], size=6, color='#888')
    ax_r.tick_params(colors='white')
    ax_r.spines['polar'].set_color('#333')
    ax_r.set_title(model, color=color, fontsize=11, fontweight='bold', pad=10)

    # RS annotation
    rs_val = rs_df[rs_df.model==model]['RS'].values[0]
    ax_r.text(0, 0.05, f'RS={rs_val:.4f}', transform=ax_r.transAxes,
              ha='left', va='bottom', color='#AAA', fontsize=7)

fig.suptitle('Family-Level Polarity Instability Radar Profiles\nEach axis = perturbation family, '
             'radius = mean flip rate', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT}/fig04_radar_profiles.png", bbox_inches='tight',
            facecolor='#1A1A2E')
plt.close()
print("  ✓ Fig 04 — Radar Spider Profiles")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 — Scatter + KDE contour: Accuracy vs RS with DR encoded as colour
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#FEFEFE')

sc = ax.scatter(rs_df['acc_original'], rs_df['RS'],
                c=rs_df['DR_degradation'], cmap='RdYlGn_r',
                s=400, zorder=5, edgecolors='#333', lw=1.5,
                norm=Normalize(vmin=0.08, vmax=0.13))

cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label('Degradation Rate (DR)', fontsize=10)

# Model labels with arrows
for _, row in rs_df.iterrows():
    ax.annotate(row['model'],
                xy=(row['acc_original'], row['RS']),
                xytext=(row['acc_original']+0.002, row['RS']+0.0008),
                fontsize=9, fontweight='bold', color='#2C3E50',
                arrowprops=dict(arrowstyle='-', color='#BBB', lw=0.8))

# Quadrant lines
ax.axhline(rs_df['RS'].mean(), color='#E74C3C', lw=1.2, ls='--', alpha=0.5)
ax.axvline(rs_df['acc_original'].mean(), color='#3498DB', lw=1.2, ls='--', alpha=0.5)

# Quadrant labels
ax.text(0.803, 0.3856, 'High acc\nHigh instability', fontsize=8,
        color='#E74C3C', ha='center', va='center', style='italic',
        bbox=dict(fc='#FFEEEE', ec='none', alpha=0.7))
ax.text(0.848, 0.3680, 'High acc\nLow instability\n(ideal)', fontsize=8,
        color='#27AE60', ha='center', va='center', style='italic',
        bbox=dict(fc='#EEFFEE', ec='none', alpha=0.7))
ax.text(0.803, 0.3829, 'Low acc\nLow instability', fontsize=8,
        color='#7F8C8D', ha='center', va='center', style='italic',
        bbox=dict(fc='#F8F8F8', ec='none', alpha=0.7))

ax.set_xlabel('Accuracy on Original Reviews', fontsize=11)
ax.set_ylabel('Robustness Score (RS) ↓ better', fontsize=11)
ax.set_title('Accuracy–Robustness Trade-off\nColour = Degradation Rate (higher DR = model degrades more under perturbation)',
             fontsize=11, pad=10)

plt.tight_layout()
plt.savefig(f"{OUT}/fig05_scatter_acc_rs_dr.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 05 — Scatter Accuracy vs RS vs DR")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 6 — Bubble chart: Dataset Vulnerability Index
#         x=n_pairs (log), y=DVI, bubble_size=DVI, colour=domain cluster
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#F0F4F8')
ax.set_facecolor('#F0F4F8')

# Domain clusters
DOMAINS = {
    'SemEval17':'Twitter','SemEval16':'Twitter','SemEval15':'Twitter',
    'SemEval13':'Twitter','SemEval18':'Twitter','ntua':'Twitter',
    'Target-dependent':'Twitter','sanders':'Twitter','hcr':'Twitter',
    'VADER':'Multi-domain','SentiSternth':'Multi-domain','sentiment140':'Twitter',
    'movie':'Reviews','hobbit':'Fan/Forum','archeage':'Fan/Forum',
    'debate08':'Political','narr':'News','irony':'Twitter',
    'iphone':'Reviews','sarcasm':'Twitter','person':'Twitter','STS-gold':'Semantic',
}
domain_colors = {
    'Twitter':'#3498DB','Reviews':'#E67E22','Fan/Forum':'#9B59B6',
    'Political':'#E74C3C','News':'#2ECC71','Multi-domain':'#F39C12',
    'Semantic':'#1ABC9C',
}

dvi_plot = dvi_df.copy()
dvi_plot['domain'] = dvi_plot['source_dataset'].map(DOMAINS).fillna('Other')

for domain in dvi_plot['domain'].unique():
    sub = dvi_plot[dvi_plot['domain']==domain]
    color = domain_colors.get(domain,'#95A5A6')
    sc = ax.scatter(np.log10(sub['n_pairs']), sub['DVI'],
                    s=sub['DVI']*800, c=color, alpha=0.75,
                    edgecolors='white', lw=1.5, label=domain, zorder=4)

for _, row in dvi_plot.iterrows():
    ax.annotate(row['source_dataset'],
                xy=(np.log10(row['n_pairs']), row['DVI']),
                xytext=(3, 3), textcoords='offset points',
                fontsize=7.5, color='#2C3E50', fontweight='bold')

ax.set_xlabel('log₁₀(Number of Perturbed Pairs)', fontsize=11)
ax.set_ylabel('Dataset Vulnerability Index (DVI)', fontsize=11)
ax.set_title('Dataset Vulnerability Index\nBubble size ∝ DVI · Colour = domain cluster · '
             'STS-gold = lowest vulnerability', fontsize=11, pad=10)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9, title='Domain')
ax.axhline(dvi_plot['DVI'].mean(), color='#7F8C8D', ls='--', lw=1.2, alpha=0.6,
           label=f'Mean DVI={dvi_plot.DVI.mean():.3f}')
ax.grid(alpha=0.3, ls='--')

plt.tight_layout()
plt.savefig(f"{OUT}/fig06_bubble_dvi.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 06 — Bubble Chart DVI")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 7 — Violin + Strip: Conditional FR distribution
#         correct_original vs incorrect_original per model
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#FAFAFA')

cond_clean = cond_df[cond_df['flip_rate'].notna()].copy()
correct   = cond_clean[cond_clean['condition']=='correct_original']
incorrect = cond_clean[cond_clean['condition']=='incorrect_original']

x_pos = np.arange(len(MODELS))
width = 0.35

# Violin for correct
for i, model in enumerate(MODELS):
    sub_c = correct[correct.model==model]['flip_rate'].dropna()
    sub_i = incorrect[incorrect.model==model]['flip_rate'].dropna()

    # Correct violin (left)
    if len(sub_c) > 1:
        parts = ax.violinplot([sub_c.values], positions=[i-0.2],
                              widths=0.35, showmedians=True,
                              showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor('#3498DB'); pc.set_alpha(0.5)
        parts['cmedians'].set_color('#2980B9'); parts['cmedians'].set_lw(2)

    # Incorrect violin (right)
    if len(sub_i) > 1:
        parts2 = ax.violinplot([sub_i.values], positions=[i+0.2],
                               widths=0.35, showmedians=True,
                               showextrema=False)
        for pc in parts2['bodies']:
            pc.set_facecolor('#E74C3C'); pc.set_alpha(0.5)
        parts2['cmedians'].set_color('#C0392B'); parts2['cmedians'].set_lw(2)

    # Strip plot on top
    jitter_c = sub_c + np.random.normal(0, 0.003, len(sub_c))
    jitter_i = sub_i + np.random.normal(0, 0.003, len(sub_i))
    ax.scatter([i-0.2]*len(sub_c), jitter_c, s=20,
               color='#2980B9', alpha=0.8, zorder=5)
    ax.scatter([i+0.2]*len(sub_i), jitter_i, s=20,
               color='#C0392B', alpha=0.8, zorder=5)

ax.set_xticks(x_pos)
ax.set_xticklabels(MODELS, fontsize=10)
ax.set_ylabel('Conditional Flip Rate', fontsize=11)
ax.set_title('Flip Rate Distributions: Correctly vs Incorrectly Classified Originals\n'
             'Blue violin = FR when original correct  ·  Red violin = FR when original incorrect',
             fontsize=11, pad=10)
ax.axhline(0.5, color='#7F8C8D', ls=':', lw=1.2, alpha=0.5, label='FR = 0.50')

blue_patch = mpatches.Patch(color='#3498DB', alpha=0.6, label='Correct original (FR_c)')
red_patch  = mpatches.Patch(color='#E74C3C', alpha=0.6, label='Incorrect original (FR_w)')
ax.legend(handles=[blue_patch, red_patch], loc='upper right', fontsize=10)
ax.grid(axis='y', ls='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/fig07_violin_conditional_fr.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 07 — Violin+Strip Conditional FR")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 8 — Connected dot plot: Cohen's h effect sizes (caterpillar)
#         Each family = one row, each model = one dot with CI
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor('#FAFAFA')

eff_valid = eff_df[eff_df['cohen_h'].notna()].copy()

y_fam = {f: i for i, f in enumerate(FAMILIES)}
model_offsets = np.linspace(-0.35, 0.35, len(MODELS))

for mi, model in enumerate(MODELS):
    sub = eff_valid[eff_valid.model==model]
    color = C_MODELS[mi]
    for _, row in sub.iterrows():
        y = y_fam[row['perturbation_family']] + model_offsets[mi]
        h = row['cohen_h']
        lo = row['cohen_h_CI_lo']
        hi = row['cohen_h_CI_hi']
        ax.plot([lo, hi], [y, y], color=color, lw=1.2, alpha=0.5)
        ax.scatter(h, y, s=50, color=color, zorder=5,
                   marker='D' if model=='ALBERT' else
                           's' if model=='BERT' else
                           '^' if model=='T5' else 'o',
                   edgecolors='white', lw=0.6, alpha=0.9)

# Reference lines
ax.axvline(0, color='#2C3E50', lw=1.5, alpha=0.6)
for ref, label in [(0.2,'small'), (0.5,'medium'), (0.8,'large')]:
    ax.axvline(ref,  color='#E74C3C', lw=0.8, ls='--', alpha=0.4)
    ax.axvline(-ref, color='#E74C3C', lw=0.8, ls='--', alpha=0.4)
    ax.text(ref+0.01, len(FAMILIES)-0.5, label, fontsize=7,
            color='#E74C3C', va='top', rotation=90)

ax.set_yticks(range(len(FAMILIES)))
ax.set_yticklabels(FAMILIES, fontsize=9)
ax.set_xlabel("Cohen's h (FR_strong − FR_mild effect size)\n0=no effect, ±0.2=small, ±0.5=medium, ±0.8=large",
              fontsize=10)
ax.set_title("Caterpillar Plot: Cohen's h Effect Sizes\n"
             "Each dot = one model·family pair · Error bars = 95% Bootstrap CI · "
             "Positive = FR increases with severity", fontsize=11, pad=10)

legend_patches = [mpatches.Patch(color=C_MODELS[i], label=m)
                  for i, m in enumerate(MODELS)]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8,
          ncol=2, framealpha=0.9)
ax.grid(axis='x', ls='--', alpha=0.2)
ax.set_xlim(-0.6, 0.65)

plt.tight_layout()
plt.savefig(f"{OUT}/fig08_caterpillar_cohens_h.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 08 — Caterpillar Cohen's h")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 9 — Sankey-style flow diagram: Position strategy → FR level
#         (manual parallel coordinates with filled polygons)
# ══════════════════════════════════════════════════════════════════════════════
pos_clean = pos_df[~pos_df['position_strategy'].str.startswith('__')].copy()
pos_clean = pos_clean.dropna(subset=['flip_rate'])

pos_means = pos_clean.groupby('position_strategy')['flip_rate'].mean().sort_values(ascending=False)
positions_ordered = pos_means.index.tolist()

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#0A0E1A')
ax.set_facecolor('#0A0E1A')

n_pos = len(positions_ordered)
n_models = len(MODELS)
x_positions = np.linspace(0, 10, n_pos)

# Draw bands per model connecting positions
for mi, model in enumerate(MODELS):
    sub_m = pos_clean[pos_clean.model==model].set_index('position_strategy')
    y_vals = []
    for pos in positions_ordered:
        if pos in sub_m.index:
            y_vals.append(sub_m.loc[pos,'flip_rate'])
        else:
            y_vals.append(np.nan)

    color = C_MODELS[mi]
    valid_mask = ~np.isnan(y_vals)
    xv = x_positions[valid_mask]
    yv = np.array(y_vals)[valid_mask]

    # Smooth line
    if len(xv) > 1:
        ax.plot(xv, yv, color=color, lw=2.0, alpha=0.8,
                marker='o', markersize=5, markeredgecolor='white',
                markeredgewidth=0.8, label=model, zorder=4)

# Position labels
for i, pos in enumerate(positions_ordered):
    label = pos.replace('-',' ').title()
    ax.text(x_positions[i], 0.275, label, rotation=45, ha='right',
            va='top', fontsize=8, color='white', fontweight='bold')
    # Vertical guide
    ax.axvline(x_positions[i], color='#333', lw=0.5, alpha=0.4)

# Colour band for high/low FR regions
ax.axhspan(0.42, 0.45, alpha=0.08, color='red')
ax.axhspan(0.34, 0.37, alpha=0.08, color='green')

ax.set_ylabel('Mean Flip Rate', color='white', fontsize=11)
ax.set_title('Parallel Profile: Flip Rate Across Grammatical Position Strategies\n'
             'Each line = one model; x-axis = position type ordered by mean FR',
             color='white', fontsize=11, pad=10)
ax.tick_params(colors='white'); ax.yaxis.label.set_color('white')
for sp in ax.spines.values(): sp.set_color('#444')
ax.set_xticks([]); ax.set_yticks(np.arange(0.32, 0.48, 0.02))
ax.yaxis.set_tick_params(labelcolor='white')
ax.legend(loc='upper right', fontsize=8, framealpha=0.2,
          labelcolor='white', facecolor='#0A0E1A', edgecolor='#444')
ax.grid(axis='y', color='#333', lw=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig(f"{OUT}/fig09_parallel_position_profiles.png", bbox_inches='tight',
            facecolor='#0A0E1A')
plt.close()
print("  ✓ Fig 09 — Parallel Coordinates Position")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 10 — Stacked area + semantic drift overlay:
#          Φ metric drift across severity × family (area chart)
#          with m7_lsd LSD shown as a contour overlay
# ══════════════════════════════════════════════════════════════════════════════
drift_m1 = drift_df[drift_df.metric=='m1_cosine_sim'].copy()
drift_lsd= drift_df[drift_df.metric=='m7_lsd'].copy()

fam_order_area = [f for f in FAMILIES
                  if drift_m1[drift_m1.perturbation_family==f]['n'].sum() > 100]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.patch.set_facecolor('#FAFAFA')

sev_colors = {'mild':'#3498DB','medium':'#F39C12','strong':'#E74C3C'}

for si, sev in enumerate(SEVS):
    ax = axes[si]
    ax.set_facecolor('#F8F8F8')

    sub_m1  = drift_m1[drift_m1.severity==sev].set_index('perturbation_family')
    sub_lsd = drift_lsd[drift_lsd.severity==sev].set_index('perturbation_family')

    m1_vals  = [sub_m1.loc[f,'mean']  if f in sub_m1.index  else np.nan for f in fam_order_area]
    lsd_vals = [sub_lsd.loc[f,'mean'] if f in sub_lsd.index else np.nan for f in fam_order_area]
    m1_std   = [sub_m1.loc[f,'std']   if f in sub_m1.index  else np.nan for f in fam_order_area]

    x = np.arange(len(fam_order_area))
    m1_arr  = np.array(m1_vals, dtype=float)
    lsd_arr = np.array(lsd_vals, dtype=float)
    std_arr = np.array(m1_std, dtype=float)

    color = sev_colors[sev]

    # Area: M1 cosine with std band
    valid = ~np.isnan(m1_arr)
    ax.fill_between(x[valid], m1_arr[valid]-std_arr[valid],
                    m1_arr[valid]+std_arr[valid],
                    alpha=0.2, color=color)
    ax.plot(x[valid], m1_arr[valid], color=color, lw=2.5,
            marker='o', markersize=5, markeredgecolor='white',
            markeredgewidth=1, label='M1 cosine (±1 SD)')

    # LSD on secondary axis
    ax2 = ax.twinx()
    valid_l = ~np.isnan(lsd_arr)
    ax2.plot(x[valid_l], lsd_arr[valid_l], color='#8E44AD', lw=2,
             ls='--', marker='s', markersize=4,
             markeredgecolor='white', markeredgewidth=0.8,
             label='M7★ LSD', alpha=0.85)
    ax2.set_ylim(-0.05, 1.15)
    ax2.set_ylabel('LSD', color='#8E44AD', fontsize=8)
    ax2.tick_params(axis='y', colors='#8E44AD', labelsize=7)
    ax2.spines['right'].set_color('#8E44AD')

    ax.set_ylabel(f'{sev.capitalize()}\nM1 Cosine Sim', fontsize=9, color=color)
    ax.set_ylim(0.70, 1.02)
    ax.set_yticks(np.arange(0.72, 1.02, 0.06))
    ax.tick_params(axis='y', labelsize=8)
    ax.axhline(0.80, color='#E74C3C', lw=0.8, ls=':', alpha=0.5)
    ax.text(len(fam_order_area)-0.5, 0.80, 'τ=0.80', color='#E74C3C',
            fontsize=7, va='bottom', ha='right')
    ax.grid(axis='both', ls='--', alpha=0.25)

    # Legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, loc='lower left',
              fontsize=7, framealpha=0.8)

axes[-1].set_xticks(np.arange(len(fam_order_area)))
axes[-1].set_xticklabels(
    [f.replace('P0','P').replace('_',' ').title() for f in fam_order_area],
    rotation=35, ha='right', fontsize=8)

fig.suptitle('Semantic Metric Drift Across Severity Levels\n'
             'M1 Cosine Similarity (area + ±1SD) and M7★ LSD (dashed) per family',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT}/fig10_area_semantic_drift.png", bbox_inches='tight')
plt.close()
print("  ✓ Fig 10 — Area Chart Semantic Drift")

print(f"\n✓ All 10 figures saved to {OUT}")
# print("\nPlot types used:")
# types = [
#     "01 — Diverging Lollipop (dual-encoded RS ranking)",
#     "02 — Annotated Dark Heatmap (FR family×model)",
#     "03 — Slopegraph (severity trend per family)",
#     "04 — Radar/Spider (12-family FR profiles per model)",
#     "05 — Scatter with colour encoding (acc vs RS vs DR)",
#     "06 — Bubble Chart (dataset vulnerability)",
#     "07 — Violin + Strip plot (conditional FR distribution)",
#     "08 — Caterpillar / Forest plot (Cohen's h + CI)",
#     "09 — Parallel Coordinates (position strategy profiles)",
#     "10 — Stacked Area + Dual-axis overlay (semantic drift)",
# ]
# for t in types: print(f"  {t}")
"""
CSPF — Replacement Plots: Fig 03 and Fig 09
Fig 03: Grouped dot matrix (Page's L significance + monotonicity per family×model)
Fig 09: Clustered position FR with CI ribbons and Kruskal-H annotation
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import FormatStrFormatter
import warnings
warnings.filterwarnings('ignore')

MODELS   = ['T5','RoBERTa','ELECTRA','XLNet','DistilBERT','GPT2','BERT','ALBERT']
FAMILIES = ['P01_hedging','P02_discourse','P03_negation','P04_sarcasm',
            'P05_dilution','P06_pragmatic','P07_contrastive','P08_synonym',
            'P09_intensity','P10_syntax','P11_entity','P12_register']
FAM_LABEL= ['P01\nHedging','P02\nDiscourse','P03\nNegation','P04\nSarcasm',
             'P05\nDilution','P06\nPragmatic','P07\nContrastive','P08\nSynonym',
             'P09\nIntensity','P10\nSyntax','P11\nEntity','P12\nRegister']
SEVS     = ['mild','medium','strong']
C_MODELS = {'T5':'#E74C3C','RoBERTa':'#E67E22','ELECTRA':'#F1C40F',
            'XLNet':'#2ECC71','DistilBERT':'#1ABC9C','GPT2':'#3498DB',
            'BERT':'#9B59B6','ALBERT':'#E91E63'}

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':10,
    'figure.dpi':150,'savefig.dpi':200,
})

# ── load ───────────────────────────────────────────────────────────────────────
mono = pd.read_csv(f"{DATA}/03_severity_monotonicity.csv")
pos  = pd.read_csv(f"{DATA}/04_position_analysis.csv")
kw   = pd.read_csv(f"{DATA}/06a_kruskal_wallis.csv")
fr   = pd.read_csv(f"{DATA}/01_flip_rate_matrix.csv")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 03 — Severity Trend Matrix
# A 12×8 annotated dot matrix showing FR_mild→FR_medium→FR_strong
# plus Page's L p-value and monotonicity flag per cell
# ══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor('#F7F9FC')

gs = gridspec.GridSpec(len(FAMILIES), len(MODELS)+1,
                        figure=fig,
                        hspace=0.08, wspace=0.05,
                        width_ratios=[1]*len(MODELS) + [0.35])

# Colour scale for severity
sev_colors = {'mild':'#3498DB','medium':'#F39C12','strong':'#E74C3C'}
# Background shade by monotone result
bg_mono  = '#E8F8E8'   # monotone = True
bg_not   = '#FFF0F0'   # monotone = False
bg_miss  = '#F5F5F5'   # no data

for fi, fam in enumerate(FAMILIES):
    for mi, model in enumerate(MODELS):
        ax = fig.add_subplot(gs[fi, mi])
        ax.set_facecolor(bg_miss)
        ax.set_xlim(0, 3)
        ax.set_ylim(0.26, 0.72)

        sub = mono[(mono.perturbation_family==fam) & (mono.model==model)]
        if sub.empty:
            ax.text(1.5, 0.50, 'N/A', ha='center', va='center',
                    fontsize=7, color='#AAA')
        else:
            row = sub.iloc[0]
            vals = {'mild': row['FR_mild'],
                    'medium': row['FR_medium'],
                    'strong': row['FR_strong']}

            is_mono = bool(row['monotone_FR']) if pd.notna(row['monotone_FR']) else False
            p_val   = float(row['pages_p'])   if pd.notna(row['pages_p'])   else 1.0
            is_sig  = bool(row['pages_sig'])  if pd.notna(row['pages_sig']) else False
            rho     = float(row['spearman_rho']) if pd.notna(row['spearman_rho']) else 0.0

            ax.set_facecolor(bg_mono if is_mono else bg_not)

            # Plot three dots connected by line
            x_coords = [0.5, 1.5, 2.5]
            y_coords = [vals.get(s, np.nan) for s in SEVS]
            valid = [not np.isnan(y) for y in y_coords]

            xv = [x for x, v in zip(x_coords, valid) if v]
            yv = [y for y, v in zip(y_coords, valid) if v]

            if len(xv) > 1:
                # Line with colour gradient mild→strong
                for k in range(len(xv)-1):
                    ax.plot([xv[k], xv[k+1]], [yv[k], yv[k+1]],
                            color='#555', lw=1.2, alpha=0.6, zorder=2)

            for xi, yi, sev in zip(x_coords, y_coords, SEVS):
                if not np.isnan(yi):
                    ax.scatter(xi, yi, s=55, color=sev_colors[sev],
                               zorder=4, edgecolors='white', lw=0.6)
                    ax.text(xi, yi + 0.022, f'{yi:.2f}',
                            ha='center', va='bottom', fontsize=5.5,
                            color='#333', fontweight='bold')

            # Page's p annotation bottom
            p_txt = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<.001'
            p_col = '#27AE60' if is_sig else '#95A5A6'
            ax.text(1.5, 0.285, p_txt, ha='center', va='bottom',
                    fontsize=5, color=p_col, fontweight='bold' if is_sig else 'normal')

            # Monotone arrow or cross
            sym = '↑' if is_mono else '≁'
            sym_col = '#27AE60' if is_mono else '#E74C3C'
            ax.text(2.8, 0.69, sym, ha='right', va='top',
                    fontsize=9, color=sym_col, fontweight='bold')

            # Spearman ρ
            ax.text(0.1, 0.69, f'ρ={rho:.2f}', ha='left', va='top',
                    fontsize=5, color='#7F8C8D')

        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor('#CCC'); sp.set_linewidth(0.5)

        # Column header (model name)
        if fi == 0:
            ax.set_title(model, fontsize=8, fontweight='bold',
                         color=C_MODELS[model], pad=4)
        # Row header
        if mi == 0:
            ax.set_ylabel(FAM_LABEL[fi], fontsize=7, rotation=0,
                          ha='right', va='center', labelpad=40,
                          color='#2C3E50', fontweight='bold')

    # Side panel: family-level FR summary bar
    ax_side = fig.add_subplot(gs[fi, len(MODELS)])
    ax_side.set_facecolor('#F0F4F8')
    ax_side.set_xlim(0, 1); ax_side.set_ylim(0, 1)
    ax_side.set_xticks([]); ax_side.set_yticks([])
    for sp in ax_side.spines.values():
        sp.set_edgecolor('#DDD'); sp.set_linewidth(0.5)

    # Kruskal H
    kw_row = kw[kw.perturbation_family == fam]
    if not kw_row.empty:
        H   = kw_row.iloc[0]['kruskal_H']
        sig = kw_row.iloc[0]['significant_0.05']
        col = '#E74C3C' if sig else '#95A5A6'
        ax_side.text(0.5, 0.6, f'H={H:.1f}', ha='center', va='center',
                     fontsize=7, color=col, fontweight='bold')
        ax_side.text(0.5, 0.25, '✓sig' if sig else 'ns',
                     ha='center', va='center', fontsize=7, color=col)

# Title
fig.text(0.5, 0.995,
         'Severity–Instability Matrix (Mild → Medium → Strong)\n'
         'Dot size fixed · Values = FR · ↑ = monotone · ρ = Spearman · '
         'p = Page\'s L · Right panel = Kruskal-Wallis H',
         ha='center', va='top', fontsize=11, fontweight='bold', color='#2C3E50')

# Legend at bottom
legend_els = [
    mpatches.Patch(color='#3498DB', label='Mild'),
    mpatches.Patch(color='#F39C12', label='Medium'),
    mpatches.Patch(color='#E74C3C', label='Strong'),
    mpatches.Patch(color='#E8F8E8', label='Monotone ↑'),
    mpatches.Patch(color='#FFF0F0', label='Non-monotone ≁'),
    mpatches.Patch(color='#27AE60', label='Page\'s L p<.05'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=6,
           fontsize=9, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.005))

plt.savefig(f"{OUT}/fig03_severity_matrix.png",
            bbox_inches='tight', facecolor='#F7F9FC', dpi=180)
plt.close()
print("✓ Fig 03 — Severity Matrix saved")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 09 — Position Strategy: Grouped Heatmap + CI ribbon + KW annotation
#
# Top panel:   annotated heatmap (position × model) with FR values
# Bottom panel: mean FR per position with 95% CI error bars + n_pairs bar
# ══════════════════════════════════════════════════════════════════════════════
pos_clean = pos[~pos['position_strategy'].str.startswith('__')].dropna(subset=['flip_rate']).copy()
kw_pos    = pos[pos['position_strategy'] == '__Kruskal_H__'].copy()

# pivot
pivot_fr = pos_clean.pivot_table(
    values='flip_rate', index='position_strategy',
    columns='model', aggfunc='mean')
pivot_n  = pos_clean.pivot_table(
    values='n_pairs', index='position_strategy',
    columns='model', aggfunc='mean')
pivot_ci_lo = pos_clean.pivot_table(
    values='CI_lower_95', index='position_strategy',
    columns='model', aggfunc='mean')
pivot_ci_hi = pos_clean.pivot_table(
    values='CI_upper_95', index='position_strategy',
    columns='model', aggfunc='mean')

# Sort positions by mean FR descending
pos_order = pivot_fr.mean(axis=1).sort_values(ascending=False).index.tolist()
pivot_fr  = pivot_fr.reindex(index=pos_order, columns=MODELS)
pivot_n   = pivot_n.reindex(index=pos_order, columns=MODELS)
pivot_ci_lo = pivot_ci_lo.reindex(index=pos_order, columns=MODELS)
pivot_ci_hi = pivot_ci_hi.reindex(index=pos_order, columns=MODELS)

n_pos = len(pos_order)

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#FAFAFA')
gs2 = gridspec.GridSpec(2, 2, figure=fig,
                         height_ratios=[1.8, 1],
                         width_ratios=[3, 1],
                         hspace=0.35, wspace=0.08)

# ── Panel A: heatmap ──────────────────────────────────────────────────────────
ax_heat = fig.add_subplot(gs2[0, 0])
ax_heat.set_facecolor('#FAFAFA')

cmap_pos = LinearSegmentedColormap.from_list(
    'pos', ['#D6EAF8','#5DADE2','#1A5276','#C0392B','#7B241C'])
vmin, vmax = 0.32, 0.48

im = ax_heat.imshow(pivot_fr.values, cmap=cmap_pos, aspect='auto',
                    vmin=vmin, vmax=vmax)

# Annotate all cells
for i in range(n_pos):
    for j in range(len(MODELS)):
        v = pivot_fr.iloc[i, j]
        lo = pivot_ci_lo.iloc[i, j]
        hi = pivot_ci_hi.iloc[i, j]
        n  = pivot_n.iloc[i, j]
        if np.isnan(v): continue
        txt_color = 'white' if v > 0.40 else '#1A1A2E'
        ax_heat.text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=8.5, color=txt_color, fontweight='bold')
        # CI as tiny sub-text
        ax_heat.text(j, i + 0.32, f'[{lo:.2f},{hi:.2f}]',
                     ha='center', va='center', fontsize=5.5,
                     color='white' if v > 0.40 else '#555')

ax_heat.set_xticks(range(len(MODELS)))
ax_heat.set_xticklabels(MODELS, rotation=30, ha='right',
                         fontsize=10, fontweight='bold')
ax_heat.set_yticks(range(n_pos))
pos_labels = [p.replace('-',' ').replace('_',' ').title() for p in pos_order]
ax_heat.set_yticklabels(pos_labels, fontsize=10)
ax_heat.set_title('Flip Rate by Grammatical Position × Model\n'
                   'Cell = FR (mean) · Sub-text = 95% Wilson CI · '
                   'Ordered by mean FR descending', fontsize=11, pad=8)

cbar = plt.colorbar(im, ax=ax_heat, shrink=0.8, pad=0.01)
cbar.set_label('Flip Rate', fontsize=9)
cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

# ── Panel B: n_pairs side bar ─────────────────────────────────────────────────
ax_n = fig.add_subplot(gs2[0, 1])
ax_n.set_facecolor('#F0F4F8')
mean_n = pivot_n.mean(axis=1)
colors_n = ['#2E86AB' if v > 1e5 else '#A8D5E2' for v in mean_n.values]
bars = ax_n.barh(range(n_pos), np.log10(mean_n.values + 1),
                  color=colors_n, edgecolor='white', lw=0.8)
ax_n.set_yticks(range(n_pos))
ax_n.set_yticklabels([])
ax_n.set_xlabel('log₁₀(mean n_pairs)', fontsize=9)
ax_n.set_title('Sample\nSize', fontsize=9, pad=4)
for i, (v, bar) in enumerate(zip(mean_n.values, bars)):
    label = f'{v/1000:.0f}K' if v >= 1000 else f'{int(v)}'
    ax_n.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
              label, va='center', fontsize=7.5, color='#333')
ax_n.spines['top'].set_visible(False)
ax_n.spines['right'].set_visible(False)
ax_n.invert_yaxis()

# ── Panel C: mean FR per position with CI + Kruskal H ────────────────────────
ax_bot = fig.add_subplot(gs2[1, :])
ax_bot.set_facecolor('#F8F9FA')

x = np.arange(n_pos)
# Per-model lines
for mi, model in enumerate(MODELS):
    y_vals = pivot_fr[model].values
    ax_bot.plot(x, y_vals, color=C_MODELS[model], lw=1.5,
                marker='o', markersize=5, label=model,
                markeredgecolor='white', markeredgewidth=0.8, alpha=0.85)

# Mean across models with CI band
grand_mean = pivot_fr.mean(axis=1).values
grand_lo   = pivot_ci_lo.mean(axis=1).values
grand_hi   = pivot_ci_hi.mean(axis=1).values
ax_bot.fill_between(x, grand_lo, grand_hi, alpha=0.15, color='#2C3E50',
                    label='Mean 95% CI band')
ax_bot.plot(x, grand_mean, color='#2C3E50', lw=2.5, ls='--',
            marker='D', markersize=7, markeredgecolor='white',
            markeredgewidth=1.2, label='Grand mean', zorder=6)

# Annotate grand mean values
for xi, ym in zip(x, grand_mean):
    ax_bot.text(xi, ym + 0.003, f'{ym:.2f}', ha='center', va='bottom',
                fontsize=8, color='#2C3E50', fontweight='bold')

# Kruskal H per model (from kw_pos)
for mi, model in enumerate(MODELS):
    kw_row = kw_pos[kw_pos.model == model]
    if not kw_row.empty:
        H   = kw_row.iloc[0]['flip_rate']   # H stored in flip_rate col
        sig = kw_row.iloc[0]['CI_lower_95'] < 0.05  # p stored in CI_lower_95
        sym = f'H={H:.0f}' + ('*' if sig else '')
        ax_bot.text(mi * (n_pos-1)/(len(MODELS)-1) if len(MODELS) > 1 else n_pos//2,
                    0.449, sym, ha='center', va='bottom',
                    fontsize=7, color=C_MODELS[model], rotation=45)

ax_bot.set_xticks(x)
ax_bot.set_xticklabels(pos_labels, rotation=35, ha='right', fontsize=9)
ax_bot.set_ylabel('Flip Rate', fontsize=10)
ax_bot.set_ylim(0.32, 0.46)
ax_bot.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax_bot.set_title('Mean Flip Rate per Position Strategy — All Models\n'
                 'Shaded band = 95% CI of grand mean · Dashed = grand mean · '
                 'Values annotated on grand mean', fontsize=10, pad=6)
ax_bot.legend(loc='upper right', fontsize=8, ncol=5,
              framealpha=0.9, edgecolor='#CCC')
ax_bot.grid(axis='y', ls='--', alpha=0.3)
ax_bot.spines['top'].set_visible(False)
ax_bot.spines['right'].set_visible(False)

# Overall title
fig.suptitle('Grammatical Position Strategy Analysis\nTop: FR heatmap with 95% CI · '
             'Right: sample sizes · Bottom: per-model FR profiles',
             fontsize=13, fontweight='bold', y=1.005, color='#1A1A2E')

plt.savefig(f"{OUT}/fig09_position_analysis.png",
            bbox_inches='tight', facecolor='#FAFAFA', dpi=180)
plt.close()
print("✓ Fig 09 — Position Analysis saved")

print("\n✓ Both replacement plots complete.")